# Voice-to-RME — Training ASR v2 (Whisper large-v3-turbo)

## CARA MENJALANKAN (baca dulu)
1. **Runtime > Change runtime type > A100 GPU.** (Jika A100 tidak tersedia, notebook tetap jalan di L4/T4 dengan profil hemat otomatis, tetapi hasil resmi memakai A100.)
2. **Runtime > Run all.**
3. Saat muncul jendela izin **Google Drive**, klik izinkan. Hanya langkah ini yang butuh klik manusia.
4. Tunggu sampai sel terakhir mencetak `STATUS_TRAINING: OK`. Setiap langkah mencetak `✅ LANGKAH n OK`.
5. Jika ada `❌ LANGKAH n GAGAL`, ikuti baris `TINDAKAN` di bawahnya. Jangan mengubah kode.
6. **Jika sesi Colab terputus:** sambungkan lagi dengan GPU yang sama, lalu **Run all** lagi. Unduhan, split, dan checkpoint training akan dilanjutkan dari Drive, tidak mulai dari nol.
7. Setelah selesai, jalankan `benchmark_v2_colab_t4.ipynb` dengan runtime **T4**.

**Hasil di Google Drive** (`MyDrive/whisper_turbo_rme_enterprise/`): `whisper_large_v3_turbo_rme_v2_merged.tar.gz`, `rme_v2_split.json`, `rme_v2_training_report.json`.

## Apa yang berubah dari v1
| | v1 | v2 |
|---|---|---|
| Split | per-klip; semua video test juga ada di train | **per-video** train/dev/test |
| Label | semua klip | buang klip >30 dtk, caption terpotong/terlalu cepat, caption berulang |
| LoRA | r=32 pada `q_proj`,`v_proj` | r=32 pada `q,k,v,out_proj,fc1,fc2` encoder+decoder |
| Regularisasi | – | SpecAugment + augmentasi audio (gain, noise, speed) |
| Pilih checkpoint | – | WER terendah pada video dev yang tidak dilihat + early stopping |

## Patch release candidate — belum divalidasi GPU
Output lama dihapus agar tidak dianggap hasil kode baru. Jalankan pada direktori Drive baru; jangan gunakan cache hipotesis lama.
Patch: bukti vital kontekstual, tidak ada regimen otomatis, preview FHIR non-transactional, gagal tertutup untuk JSON/gerbang rilis.
Lulus benchmark bukan sertifikasi enterprise. Lihat AUDIT_RELEASE.md di paket.


## LANGKAH 1/10 — Pasang dependensi
- **Tujuan:** memasang pustaka yang dibutuhkan
- **Perkiraan waktu:** ±2 menit
- **Berhasil jika:** `✅ LANGKAH 1 OK`
- **Jika gagal:** jika Colab menampilkan tombol **Restart session**, klik lalu **Run all** lagi

In [ ]:
class LangkahGagal(RuntimeError):
    pass   # any exception halts Run all, so later steps never run on a broken state

def ok(n, msg=""):
    print(f"✅ LANGKAH {n} OK" + (f" — {msg}" if msg else ""))

def fail(n, why, action):
    raise LangkahGagal(f"❌ LANGKAH {n} GAGAL: {why} | TINDAKAN: {action}")
!pip install -q "transformers>=4.46" accelerate peft "torchao>=0.16" bitsandbytes librosa soundfile pandas matplotlib tqdm
import importlib
for m in ("transformers", "peft", "accelerate", "librosa", "soundfile", "pandas"):
    importlib.import_module(m)
ok(1)

## LANGKAH 2/10 — Cek GPU dan pilih profil training
- **Tujuan:** memastikan GPU aktif dan memilih profil otomatis
- **Perkiraan waktu:** <1 menit
- **Berhasil jika:** `✅ LANGKAH 2 OK — profil ...`
- **Jika gagal:** aktifkan GPU di Runtime > Change runtime type, lalu Run all

In [ ]:
import torch, os, json
if not torch.cuda.is_available():
    fail(2, "GPU tidak aktif", "Runtime > Change runtime type > GPU (A100/L4/T4), lalu Run all")
GPU = torch.cuda.get_device_name(0); MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
TRAIN_PROFILE = "auto"     # "auto" | "full" | "hemat"  (biarkan "auto")
profile = TRAIN_PROFILE if TRAIN_PROFILE != "auto" else ("hemat" if MEM_GB < 30 else "full")
if profile == "full":
    EPOCHS, EVAL_STEPS, DEV_EVAL_CLIPS = 3, 250, 600
else:   # T4 16 GB: 1 epoch, eval lebih jarang & lebih kecil -> muat dalam satu sesi Colab
    EPOCHS, EVAL_STEPS, DEV_EVAL_CLIPS = 1, 150, 300
BATCH, ACCUM = (32, 1) if MEM_GB >= 70 else (16, 2) if MEM_GB >= 35 else (8, 4)   # batch efektif 32
print(GPU, f"{MEM_GB:.0f} GB | bf16: {torch.cuda.is_bf16_supported()}")
ok(2, f"profil {profile}: {EPOCHS} epoch, batch {BATCH}x{ACCUM}")

## LANGKAH 3/10 — Hubungkan Google Drive dan konfigurasi
- **Tujuan:** tempat menyimpan checkpoint, split, dan model
- **Perkiraan waktu:** <1 menit (+klik izin Drive)
- **Berhasil jika:** `✅ LANGKAH 3 OK`
- **Jika gagal:** klik izin Drive saat diminta; jika ditolak, jalankan ulang sel ini

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/whisper_turbo_rme_enterprise"; os.makedirs(DRIVE_DIR, exist_ok=True)
SPLIT_SEED, DEV_FRAC, TEST_FRAC = 13, 0.05, 0.10      # JANGAN diubah: benchmark memakai split yang sama
SPLIT_PATH = f"{DRIVE_DIR}/rme_v2_split.json"
CKPT_DIR = f"{DRIVE_DIR}/v2_rc_checkpoints_{profile}"
MERGED_DIR = "/content/whisper_large_v3_turbo_rme_v2_rc_merged"
LR, LORA_R, LORA_ALPHA, PATIENCE = 1e-4, 32, 64, 4
WORKERS = min(8, os.cpu_count() or 2)
MAX_STEPS = -1
os.chdir("/content")
ok(3, DRIVE_DIR)

## LANGKAH 4/10 — Unduh dataset (Level 2 + Level 3, ±1,4 GB)
- **Tujuan:** mengunduh manifest dan audio, lalu mengekstrak
- **Perkiraan waktu:** ±10 menit
- **Berhasil jika:** `✅ LANGKAH 4 OK — ... klip audio siap`
- **Jika gagal:** ikuti TINDAKAN yang dicetak

In [ ]:
BASE = "https://github.com/faisal-evos/voice-rme-data/releases/download"
os.makedirs("data/all_audio", exist_ok=True)
for rel in ["v2.0/grand_train_manifest.json", "v2.0/grand_test_manifest.json",
            "v1.0/clean_train_dataset.tar.gz", "v1.0/clean_test_dataset.tar.gz",
            "v2.0/level3_audio_part1.tar.gz", "v2.0/level3_audio_part2.tar.gz", "v2.0/level3_audio_part3.tar.gz"]:
    fn = os.path.basename(rel)
    if not os.path.exists(fn):
        !wget -c -q --show-progress {BASE}/{rel}
    if not os.path.exists(fn):
        fail(4, f"unduhan {fn} gagal", "periksa koneksi internet Colab lalu jalankan ulang sel ini")
    if fn.endswith(".tar.gz") and not os.path.exists(fn + ".done"):
        !tar xzf {fn} -C data/all_audio && touch {fn}.done
need = {x["audio_file"] for f in ("grand_train_manifest.json", "grand_test_manifest.json") for x in json.load(open(f))}
have = set(os.listdir("data/all_audio"))
cover = len(need & have) / len(need)
print(f"file audio: {len(have)} | cakupan manifest: {cover:.1%}")
if cover < 0.95:
    fail(4, f"hanya {cover:.1%} audio manifest tersedia", "hapus file *.tar.gz dan *.done di /content lalu jalankan ulang sel ini")
ok(4, f"{len(need & have)} klip audio siap")

## LANGKAH 5/10 — Tulis modul dan uji logika
- **Tujuan:** menulis kode resep v2 lalu menjalankan tes CPU
- **Perkiraan waktu:** <1 menit
- **Berhasil jika:** `✅ LANGKAH 5 OK`
- **Jika gagal:** jangan lanjut; kirim output sel tes ke pemilik proyek

In [ ]:
%%writefile clinical_safety.py
"""Conservative numeric evidence matching; unsupported/ambiguous values stay absent.
This is a precision-first draft guard, not a clinical truth validator. Number words,
negated/historical contexts require human review and may be rejected.
"""
import re
import math

PATTERNS = {
    'blood_pressure': r'\b(?:tensi|tekanan darah|td|bp)\s*[:=]?\s*(\d{2,3})\s*(?:/|per)\s*(\d{2,3})(?![\d.,])',
    'heart_rate': r'\b(?:nadi|hr|heart rate|detak jantung)\s*[:=]?\s*(\d{2,3})(?![\d.,])',
    'respiratory_rate': r'\b(?:rr|frekuensi napas|frekuensi pernapasan|laju napas)\s*[:=]?\s*(\d{1,2})(?![\d.,])',
    'temperature': r'\b(?:suhu|temperatur|temp)\s*[:=]?\s*(\d{2}(?:[.,]\d{1,2})?)(?![\d.,])',
    'spo2': r'\b(?:spo2|saturasi)\s*[:=]?\s*(\d{2,3})(?![\d.,])',
}
RANGES = {'systolic': (40, 300), 'diastolic': (20, 200), 'heart_rate': (20, 250),
          'respiratory_rate': (4, 80), 'temperature': (25, 45), 'spo2': (50, 100)}


def extract_supported_vitals(text):
    """Require named measurement, unique value, and plausible range; reject ambiguity."""
    found = {k: set() for k in PATTERNS}
    # Sentence/turn boundary; do not split decimal punctuation.
    for clause in re.split(r'[;!?\n]|(?<!\d)\.(?!\d)', text.lower()):
        if re.search(r'\b(tidak|bukan|belum|jangan|riwayat|kemarin|contoh|misalnya|target)\b', clause):
            continue
        for key, pattern in PATTERNS.items():
            for m in re.finditer(pattern, clause):
                found[key].add(tuple(float(x.replace(',', '.')) for x in m.groups()))
    out = {}
    for key, values in found.items():
        if len(values) != 1:
            continue
        value = next(iter(values))
        if key == 'blood_pressure':
            sys_, dia = value
            if RANGES['systolic'][0] <= sys_ <= RANGES['systolic'][1] and RANGES['diastolic'][0] <= dia <= RANGES['diastolic'][1] and sys_ > dia:
                out.update(systolic=int(sys_), diastolic=int(dia))
        elif RANGES[key][0] <= value[0] <= RANGES[key][1]:
            out[key] = value[0] if key == 'temperature' else int(value[0])
    return out


def filter_vitals(proposed, transcript):
    supported = extract_supported_vitals(transcript)
    accepted, rejected = {}, []
    for key in RANGES:
        value = proposed.get(key)
        if value is None:
            continue
        try:
            number = float(str(value).replace(',', '.'))
            valid = not isinstance(value, bool) and math.isfinite(number) and number == supported.get(key)
        except (ValueError, TypeError, OverflowError):
            valid = False
        if valid:
            accepted[key] = supported[key]
        else:
            rejected.append(key)
    # Blood pressure is an atomic pair.
    if ('systolic' in accepted) != ('diastolic' in accepted):
        for key in ('systolic', 'diastolic'):
            if key in accepted:
                accepted.pop(key); rejected.append(key)
    return accepted, rejected


def validate_payload(data):
    """Validate shape before grounding; do not silently coerce malformed clinical entities."""
    if not isinstance(data, dict):
        raise ValueError('SOAP must be an object')
    for key in ('diagnoses', 'medications', 'procedures'):
        values = data.get(key)
        if values is None:
            continue
        if not isinstance(values, list):
            raise ValueError(f'{key} must be a list')
        for item in values:
            if not isinstance(item, dict) or not isinstance(item.get('term'), str) or not item['term'].strip():
                raise ValueError(f'{key} item requires a nonempty term')
    if data.get('vitals') is not None and not isinstance(data['vitals'], dict):
        raise ValueError('vitals must be an object')
    if data.get('speakers') is not None and not isinstance(data['speakers'], dict):
        raise ValueError('speakers must be an object')
    for key in ('chief_complaint', 'history_of_present_illness', 'past_medical_history',
                'allergies', 'physical_exam', 'patient_education', 'follow_up_plan'):
        if data.get(key) is not None and not isinstance(data[key], str):
            raise ValueError(f'{key} must be a string or null')


In [ ]:
%%writefile release_policy.py
"""Fail-closed benchmark verdict. No benchmark alone certifies enterprise release."""
REQUIRED = frozenset(f'G{i}' for i in range(1, 8))


def release_verdict(gates):
    ids = [g['gerbang'] for g in gates]
    if len(ids) != len(set(ids)):
        raise ValueError('Duplicate gate IDs')
    by_id = {g['gerbang']: g['status'] for g in gates}
    failed = sorted(k for k, v in by_id.items() if v == 'GAGAL')
    if failed:
        return 'BELUM LAYAK RILIS: ' + ', '.join(failed) + ' gagal'
    missing = sorted(k for k in REQUIRED if by_id.get(k) != 'LULUS')
    if missing:
        return 'BELUM DAPAT DIPUTUSKAN: data kurang untuk ' + ', '.join(missing)
    return ('LULUS GERBANG BENCHMARK; kandidat uji klinis terkendali. '
            'Rilis enterprise menunggu validasi klinis, keamanan, operasional dan integrasi.')


In [ ]:
%%writefile test_release_safety.py
from clinical_safety import extract_supported_vitals, filter_vitals
from release_policy import release_verdict
from local_med_llm import VitalSigns, SovereignLocalMedicalLLM, DialogueTurn
from live_pipeline_t4 import ground_soap, build_fhir_draft
from benchmark_real import vitals_unsupported, paired_bootstrap
from rme_asr_v2 import corpus_wer

META=dict(patient_id='P1', patient_name='X', doctor_name='Y', encounter_id='E1')

def test_vitals_no_default_or_number_collision():
    assert all(v is None for v in vars(VitalSigns()).values())
    assert not extract_supported_vitals('usia 80 tahun, berat 120 kg, tinggi 170 cm')
    assert SovereignLocalMedicalLLM().extract_vitals('tinggi 170').blood_pressure is None
    v, bad=filter_vitals({'heart_rate':80,'systolic':120,'diastolic':80}, 'usia 80 tahun berat 120 kg')
    assert not v and len(bad)==3

def test_vitals_context_negation_ambiguity_and_decimals():
    assert extract_supported_vitals('tekanan darah 130/85; nadi 80; suhu 36,7; rr 18; saturasi 98') == dict(systolic=130,diastolic=85,heart_rate=80,temperature=36.7,respiratory_rate=18,spo2=98)
    assert not extract_supported_vitals('nadi 80; nadi 90')
    assert not extract_supported_vitals('bukan nadi 80; suhu 90')
    assert filter_vitals({'heart_rate':True,'temperature':'NaN'}, 'nadi 80; suhu 36.7')[0]=={}
    assert not extract_supported_vitals('nadi 8000; suhu 36.789')

def test_live_drops_unsupported_vitals_and_regimen_defaults():
    turns=[DialogueTurn(1,'UNKNOWN',0,5,'konjungtivitis, kloramfenikol tetes mata')]
    data={'vitals':{'heart_rate':80},'medications':[{'term':'kloramfenikol','evidence_turn':1}], 'diagnoses':[{'term':'konjungtivitis','evidence_turn':1}]}
    out=ground_soap(data,turns,META); soap=out['soap']
    assert soap.vitals.heart_rate is None and out['needs_review']
    med=soap.medications[0]
    assert med.dosage==med.frequency=='Tidak disebutkan' and med.duration_days==0
    bundle=build_fhir_draft(soap)
    assert bundle['type']=='collection'
    assert all('request' not in e for e in bundle['entry'])
    assert not any(e['resource']['resourceType'] in ('MedicationRequest','Procedure','Observation') for e in bundle['entry'])

def test_benchmark_vitals_requires_type_context():
    soap=ground_soap({},[],META)['soap']; soap.vitals.heart_rate=80
    assert vitals_unsupported(soap,'usia 80')==['heart_rate']
    assert vitals_unsupported(soap,'nadi 80')==[]

def test_release_policy_failure_missing_and_unknown():
    gates=[dict(gerbang=f'G{i}',status='LULUS') for i in range(1,8)]
    assert 'LULUS GERBANG' in release_verdict(gates)
    gates[-1]['status']='GAGAL'
    assert 'BELUM LAYAK' in release_verdict(gates)
    gates[-1]['status']='N/A'
    assert 'BELUM DAPAT' in release_verdict(gates)
    assert 'BELUM DAPAT' in release_verdict([])
    try: release_verdict(gates+[gates[0]])
    except ValueError: pass
    else: raise AssertionError('duplicate gates accepted')

def test_alignment_errors_rejected():
    try: paired_bootstrap([dict(ref='a',hyp='a')],[dict(ref='b',hyp='b')])
    except ValueError: pass
    else: raise AssertionError('mismatched refs accepted')
    try: corpus_wer(['a'],[])
    except ValueError: pass
    else: raise AssertionError('truncated hypotheses accepted')

def test_cluster_bootstrap_resamples_recordings():
    a=[dict(chunk_id=str(i),video_id=str(i//2),ref='mata merah',hyp='mata') for i in range(8)]
    b=[dict(x,hyp='mata merah') for x in a]
    r=paired_bootstrap(a,b,B=200)
    assert r['n_clusters']==4 and r['bootstrap_unit']=='video'
    assert r['delta_pp']==50 and r['delta_ci95']==[50,50]

def test_invalid_clinical_shapes_blocked():
    from clinical_safety import validate_payload
    for payload in [[], {'vitals': '120/80'}, {'diagnoses': 'demam'},
                    {'medications': [None]}, {'diagnoses': [{'term': ''}]},
                    {'chief_complaint': ['nyeri']}]:
        try: validate_payload(payload)
        except ValueError: pass
        else: raise AssertionError(f'invalid clinical shape accepted: {payload}')


In [ ]:
%%writefile local_med_llm.py
#!/usr/bin/env python3
"""
Sovereign Local Medical LLM Engine for Indonesian Healthcare (Voice-to-RME)
=============================================================================
Integration targets (compliance has not been validated):
- Permenkes No. 24/2022 (SATUSEHAT HL7 FHIR R4 Electronic Medical Records)
- UU Perlindungan Data Pribadi (UU PDP No. 27/2022 - On-Premises Zero Egress)

Features:
1. Clinical Dialogue Turn Segmentation & Speaker Attribution (Dokter vs Pasien)
2. Indonesian Clinical Entity Extraction (ICD-10, ICD-9-CM, KFA Kemenkes, LOINC)
3. Bidirectional Linked Evidence Traceability (Audio Millisecond Timestamps)
4. Automated SOAP Clinical Note Generation (Subjective, Objective, Assessment, Plan)
5. SATUSEHAT HL7 FHIR R4 Transaction Bundle Builder
6. Offline Sovereign Clinical Reasoner with extensible Local LLM interface (Ollama / HuggingFace)
"""

import os
import re
import json
import time
from typing import Any, Dict, List, Optional, Tuple, Union
from dataclasses import dataclass, field, asdict

# ---------------------------------------------------------------------------
# Data Models for Clinical Traceability & Linked Evidence
# ---------------------------------------------------------------------------

@dataclass
class LinkedEvidence:
    claim_text: str
    source_speaker: str       # "DOKTER" or "PASIEN"
    audio_start_ms: int
    audio_end_ms: int
    confidence: float = 0.95
    quote_snippet: str = ""

@dataclass
class VitalSigns:
    blood_pressure: Optional[str] = None
    systolic_bp: Optional[int] = None
    diastolic_bp: Optional[int] = None
    heart_rate: Optional[int] = None
    respiratory_rate: Optional[int] = None
    temperature: Optional[float] = None
    spo2: Optional[int] = None
    gds: Optional[int] = None  # Gula Darah Sewaktu (mg/dL)

@dataclass
class DiagnosisItem:
    description: str
    icd10_code: str
    icd10_display: str
    is_primary: bool = False
    evidence: Optional[LinkedEvidence] = None

@dataclass
class MedicationItem:
    drug_name: str
    kfa_code: str
    dosage: str
    frequency: str
    route: str
    duration_days: int
    instructions: str
    evidence: Optional[LinkedEvidence] = None

@dataclass
class ProcedureItem:
    procedure_name: str
    icd9cm_code: str
    icd9cm_display: str
    evidence: Optional[LinkedEvidence] = None

@dataclass
class DialogueTurn:
    turn_id: int
    speaker: str  # "DOKTER" or "PASIEN"
    start: float  # seconds
    end: float    # seconds
    text: str

@dataclass
class SOAPRecord:
    patient_id: str
    patient_name: str
    doctor_name: str
    encounter_id: str
    specialty: str

    # S - Subjective
    chief_complaint: str
    history_of_present_illness: str
    past_medical_history: str
    allergies: str
    subjective_narrative: str

    # O - Objective
    vitals: VitalSigns
    physical_exam: str
    objective_narrative: str

    # A - Assessment
    assessment_narrative: str
    diagnoses: List[DiagnosisItem]

    # P - Plan
    plan_narrative: str
    medications: List[MedicationItem]
    procedures: List[ProcedureItem]
    patient_education: str
    follow_up_plan: str

    created_at: str = field(default_factory=lambda: time.strftime("%Y-%m-%dT%H:%M:%S+07:00"))


# ---------------------------------------------------------------------------
# Comprehensive Indonesian Clinical Ontology (ICD-10, ICD-9-CM, KFA, LOINC)
# ---------------------------------------------------------------------------

CLINICAL_ONTOLOGY = {
    # 1. Oftalmologi (Mata)
    "mata": {
        "specialty": "Spesialis Mata (Oftalmologi)",
        "diagnoses": [
            {"keywords": ["konjungtivitis", "mata merah", "perih", "belek", "belekan"],
             "code": "H10.1", "display": "Acute atopic conjunctivitis", "desc": "Konjungtivitis Akut Mata Kanan/Kiri"},
            {"keywords": ["epifora", "berair", "air mata berlebih", "hiperlakrimasi"],
             "code": "H04.2", "display": "Epiphora", "desc": "Epifora / Hiperlakrimasi"},
            {"keywords": ["ulkus kornea", "kornea keruh", "luka kornea"],
             "code": "H16.0", "display": "Corneal ulcer, unspecified", "desc": "Ulkus Kornea"},
            {"keywords": ["glaukoma", "tekanan bola mata", "mata tegang"],
             "code": "H40.1", "display": "Primary open-angle glaucoma", "desc": "Glaukoma Sudut Terbuka Primer"},
            {"keywords": ["katarak", "keruh", "pandangan berkabut"],
             "code": "H26.9", "display": "Cataract, unspecified", "desc": "Katarak Senilis / Insipien"},
            {"keywords": ["mata kering", "dry eye", "mengganjal pasir"],
             "code": "H04.1", "display": "Other disorders of lacrimal gland (Dry eye)", "desc": "Dry Eye Syndrome"}
        ],
        "medications": [
            {"keywords": ["kloramfenikol", "chloramphenicol", "tetes mata kloram"],
             "name": "Kloramfenikol Tetes Mata 0.5%", "kfa": "93000123", "dose": "1 tetes", "freq": "4 kali sehari (tiap 6 jam)", "route": "Ophthalmic", "days": 5, "inst": "Teteskan pada forniks konjungtiva bawah, jangan sentuh ujung botol."},
            {"keywords": ["cendofresh", "tetes mata pelembab", "air mata buatan", "pelembab mata", "artificial tears"],
             "name": "Carboxymethylcellulose 0.5% (Tetes Mata)", "kfa": "93000456", "dose": "1 tetes", "freq": "3-4 kali sehari prn", "route": "Ophthalmic", "days": 14, "inst": "Teteskan saat terasa kering atau mengganjal."},
            {"keywords": ["tobramisin", "tobramycin", "dexamethasone"],
             "name": "Tobramycin 0.3% + Dexamethasone 0.1% Tetes Mata", "kfa": "93000789", "dose": "1 tetes", "freq": "3 kali sehari", "route": "Ophthalmic", "days": 5, "inst": "Kocok dahulu sebelum digunakan."},
            {"keywords": ["natrium diklofenak", "anti radang mata"],
             "name": "Natrium Diklofenak 0.1% Tetes Mata", "kfa": "93000234", "dose": "1 tetes", "freq": "3 kali sehari", "route": "Ophthalmic", "days": 5, "inst": "Gunakan setelah makan jika ada anti nyeri oral."}
        ],
        "procedures": [
            {"keywords": ["slit lamp", "lampu celah", "biomikroskopi"],
             "code": "95.02", "display": "Slit lamp biomicroscopy", "desc": "Pemeriksaan Slit Lamp & Segmen Anterior Mata"},
            {"keywords": ["funduskopi", "oftalmoskopi", "retina"],
             "code": "95.04", "display": "Ophthalmoscopy", "desc": "Pemeriksaan Funduskopi Direk/Indirek"},
            {"keywords": ["tonometri", "tekanan bola mata", "tonometer"],
             "code": "95.26", "display": "Tonometry", "desc": "Pemeriksaan Tonometri Schiotz/Non-Contact"},
            {"keywords": ["fluorescein", "tes fluoresensi", "kertas fluoresin"],
             "code": "95.03", "display": "Fluorescein dye test of eye", "desc": "Tes Fluorescein Kornea"}
        ]
    },

    # 2. Neurologi (Saraf)
    "saraf": {
        "specialty": "Spesialis Saraf (Neurologi)",
        "diagnoses": [
            {"keywords": ["stroke", "stroke iskemik", "infark", "lemah separuh", "pelo", "hemiparesis"],
             "code": "I63.9", "display": "Cerebral infarction, unspecified", "desc": "Stroke Iskemik Akut / Infark Serebri"},
            {"keywords": ["stroke hemoragik", "perdarahan otak"],
             "code": "I61.9", "display": "Nontraumatic intracerebral hemorrhage", "desc": "Stroke Hemoragik"},
            {"keywords": ["babinski", "refleks patologis", "paresis n.vii", "pelo tiba-tiba"],
             "code": "R29.2", "display": "Abnormal reflex (Babinski sign positive)", "desc": "Refleks Patologis Babinski Positif"},
            {"keywords": ["tension headache", "sakit kepala tegang", "terikat kencang"],
             "code": "G44.2", "display": "Tension-type headache", "desc": "Tension Type Headache (TTH)"},
            {"keywords": ["migrain", "sakit kepala sebelah", "denyut"],
             "code": "G43.9", "display": "Migraine, unspecified", "desc": "Migrain Tanpa Aura"},
            {"keywords": ["bell's palsy", "wajah mencong", "paresis fasialis"],
             "code": "G51.0", "display": "Bell's palsy", "desc": "Bell's Palsy (Paresis N. VII Perifer)"}
        ],
        "medications": [
            {"keywords": ["aspirin", "asetosal", "aspilet", "asam asetilsalisilat"],
             "name": "Aspilets (Aspirin) 80 mg Tablet", "kfa": "93001101", "dose": "1 tablet (80 mg)", "freq": "1 kali sehari", "route": "Oral", "days": 30, "inst": "Diminum sesudah makan pagi."},
            {"keywords": ["clopidogrel", "klopidogrel"],
             "name": "Clopidogrel 75 mg Tablet", "kfa": "93001102", "dose": "1 tablet (75 mg)", "freq": "1 kali sehari", "route": "Oral", "days": 30, "inst": "Diminum pada jam yang sama setiap hari."},
            {"keywords": ["citicoline", "sitikolin", "neuroprotektor"],
             "name": "Citicoline 500 mg Tablet", "kfa": "93001103", "dose": "1 tablet (500 mg)", "freq": "2 kali sehari", "route": "Oral", "days": 14, "inst": "Diminum sebelum atau sesudah makan."},
            {"keywords": ["atorvastatin", "statin", "kolesterol"],
             "name": "Atorvastatin 20 mg Tablet", "kfa": "93001104", "dose": "1 tablet (20 mg)", "freq": "1 kali sehari (malam)", "route": "Oral", "days": 30, "inst": "Diminum malam hari menjelang tidur."},
            {"keywords": ["mecobalamin", "metilkobalamin", "vitamin b12 saraf"],
             "name": "Mecobalamin 500 mcg Kapsul", "kfa": "93001105", "dose": "1 kapsul (500 mcg)", "freq": "3 kali sehari", "route": "Oral", "days": 14, "inst": "Diminum sesudah makan."}
        ],
        "procedures": [
            {"keywords": ["ct scan", "ct kepala", "neuroimaging"],
             "code": "87.03", "display": "Computerized axial tomography of head", "desc": "CT Scan Kepala Non-Kontras"},
            {"keywords": ["refleks", "babinski", "motorik", "pemeriksaan fisik saraf"],
             "code": "93.01", "display": "Functional evaluation (Neurological exam)", "desc": "Pemeriksaan Neurologis Lengkap & Refleks Babinski"},
            {"keywords": ["eeg", "elektroensefalografi"],
             "code": "89.14", "display": "Electroencephalogram", "desc": "Pemeriksaan EEG"},
            {"keywords": ["fisioterapi", "rehabilitasi medik"],
             "code": "93.11", "display": "Physical therapy exercises", "desc": "Konsul Rehabilitasi Medik & Fisioterapi Motorik"}
        ]
    },

    # 3. Kardiologi (Jantung & Hipertensi)
    "jantung": {
        "specialty": "Spesialis Jantung & Pembuluh Darah (Kardiologi)",
        "diagnoses": [
            {"keywords": ["hipertensi", "tensi tinggi", "darah tinggi", "160/90", "150/90", "170/100"],
             "code": "I10", "display": "Essential (primary) hypertension", "desc": "Hipertensi Primer Derajat II"},
            {"keywords": ["angina", "nyeri dada", "menjalar ke punggung", "menjalar ke rahang", "dada ditekan beban", "iskemik"],
             "code": "I20.0", "display": "Unstable angina pectoris", "desc": "Angina Pektoris Tidak Stabil (UAP)"},
            {"keywords": ["infark miokard", "serangan jantung", "stemi", "nstemi"],
             "code": "I21.9", "display": "Acute myocardial infarction, unspecified", "desc": "Sindrom Koroner Akut (SKA)"},
            {"keywords": ["gagal jantung", "sesak napas malam", "bengkak kaki", "dekompensasi kordis"],
             "code": "I50.9", "display": "Heart failure, unspecified", "desc": "Gagal Jantung Kongestif (CHF NYHA II/III)"}
        ],
        "medications": [
            {"keywords": ["amlodipine", "amlodipin"],
             "name": "Amlodipine 10 mg Tablet", "kfa": "93002202", "dose": "1 tablet (10 mg)", "freq": "1 kali sehari (pagi)", "route": "Oral", "days": 30, "inst": "Diminum teratur tiap pagi."},
            {"keywords": ["candesartan", "kandesartan"],
             "name": "Candesartan 16 mg Tablet", "kfa": "93002203", "dose": "1 tablet (16 mg)", "freq": "1 kali sehari", "route": "Oral", "days": 30, "inst": "Diminum malam hari atau pagi hari."},
            {"keywords": ["isdn", "isosorbid dinitrat"],
             "name": "Isosorbide Dinitrate 5 mg Sublingual", "kfa": "93002201", "dose": "1 tablet (5 mg)", "freq": "Bila dada nyeri (prn)", "route": "Sublingual", "days": 10, "inst": "Letakkan di bawah lidah saat dada terasa tertekan nyeri."},
            {"keywords": ["bisoprolol", "beta blocker"],
             "name": "Bisoprolol 2.5 mg Tablet", "kfa": "93002204", "dose": "1 tablet (2.5 mg)", "freq": "1 kali sehari (pagi)", "route": "Oral", "days": 30, "inst": "Diminum pagi hari sesudah sarapan."},
            {"keywords": ["furosemid", "furosemide"],
             "name": "Furosemide 40 mg Tablet", "kfa": "93002205", "dose": "1 tablet (40 mg)", "freq": "1 kali sehari (pagi)", "route": "Oral", "days": 7, "inst": "Diminum pagi hari untuk membuang cairan bengkak."}
        ],
        "procedures": [
            {"keywords": ["ekg", "rekam jantung", "elektrokardiogram"],
             "code": "89.52", "display": "Electrocardiogram", "desc": "Perekaman EKG 12 Sandapan"},
            {"keywords": ["rontgen thorax", "foto dada", "toraks"],
             "code": "87.44", "display": "Routine chest x-ray", "desc": "Foto Rontgen Thorax PA (Cardiothoracic Ratio)"},
            {"keywords": ["ekokardiografi", "echo jantung", "usg jantung"],
             "code": "88.72", "display": "Diagnostic ultrasound of heart (Echocardiography)", "desc": "Pemeriksaan Ekokardiografi Transtorakal"}
        ]
    },

    # 4. Penyakit Dalam / Endokrin / Lambung (Interna)
    "interna": {
        "specialty": "Spesialis Penyakit Dalam (Interna)",
        "diagnoses": [
            {"keywords": ["diabetes", "gula darah", "kencing manis", "sering haus", "dm tipe 2"],
             "code": "E11.9", "display": "Type 2 diabetes mellitus without complications", "desc": "Diabetes Melitus Tipe 2 Terkontrol Sebagian"},
            {"keywords": ["dispepsia", "maag", "perut perih", "kembung", "begah", "ulu hati"],
             "code": "K30", "display": "Dyspepsia", "desc": "Dispepsia Fungsional / Sindrom Dispepsia"},
            {"keywords": ["gerd", "asam lambung", "heartburn", "dada panas", "panas tenggorokan"],
             "code": "K21.9", "display": "Gastro-esophageal reflux disease without esophagitis", "desc": "Gastroesophageal Reflux Disease (GERD)"},
            {"keywords": ["diare", "gastroenteritis", "mencret", "buang air cair"],
             "code": "A09", "display": "Infectious gastroenteritis and colitis, unspecified", "desc": "Gastroenteritis Akut (GEA)"}
        ],
        "medications": [
            {"keywords": ["metformin", "metformin 500"],
             "name": "Metformin HCl 500 mg Tablet", "kfa": "93003301", "dose": "1 tablet (500 mg)", "freq": "2 kali sehari", "route": "Oral", "days": 30, "inst": "Diminum bersamaan atau segera sesudah makan."},
            {"keywords": ["glimepiride", "glimepirid"],
             "name": "Glimepiride 2 mg Tablet", "kfa": "93003302", "dose": "1 tablet (2 mg)", "freq": "1 kali sehari", "route": "Oral", "days": 30, "inst": "Diminum pagi hari 15 menit sebelum sarapan."},
            {"keywords": ["omeprazole", "omeprazol"],
             "name": "Omeprazole 20 mg Kapsul", "kfa": "93003303", "dose": "1 kapsul (20 mg)", "freq": "2 kali sehari", "route": "Oral", "days": 10, "inst": "Diminum 30 menit sebelum makan."},
            {"keywords": ["sukralfat", "sucralfate"],
             "name": "Sucralfate Suspensi 100 ml", "kfa": "93003304", "dose": "1 sendok makan (15 ml)", "freq": "3 kali sehari", "route": "Oral", "days": 7, "inst": "Diminum 1 jam sebelum makan atau 2 jam sesudah makan."},
            {"keywords": ["antasida", "antasid"],
             "name": "Antasida DOEN Tablet Kunyah", "kfa": "93003305", "dose": "1 tablet", "freq": "3 kali sehari kunyah", "route": "Oral", "days": 5, "inst": "Dikunyah 1 jam sebelum makan atau saat nyeri ulu hati."}
        ],
        "procedures": [
            {"keywords": ["gula darah", "gds", "hba1c", "cek gula"],
             "code": "90.59", "display": "Microscopic examination of blood", "desc": "Pemeriksaan GDS, GDP, dan HbA1c"},
            {"keywords": ["usg abdomen", "usg perut"],
             "code": "88.76", "display": "Diagnostic ultrasound of abdomen", "desc": "USG Abdomen Atas & Bawah"},
            {"keywords": ["endoskopi", "gastroskopi"],
             "code": "45.13", "display": "Esophagogastroduodenoscopy", "desc": "Endoskopi Saluran Cerna Atas (EGD)"}
        ]
    },

    # 5. Obstetri & Ginekologi (Kandungan / Obgyn)
    "kandungan": {
        "specialty": "Spesialis Kebidanan & Kandungan (Obgyn)",
        "diagnoses": [
            {"keywords": ["hamil", "antenatal", "anc", "pemeriksaan kehamilan", "kandungan"],
             "code": "Z34.9", "display": "Supervision of normal pregnancy, unspecified", "desc": "Pemeriksaan Kehamilan Normal (ANC Trimester II)"},
            {"keywords": ["infertilitas", "infertility", "susah hamil", "program hamil", "promil"],
             "code": "N97.9", "display": "Female infertility, unspecified", "desc": "Infertilitas Primer / Konseling Pra-Konsepsi"},
            {"keywords": ["hiperemesis", "mual muntah hamil"],
             "code": "O21.0", "display": "Mild hyperemesis gravidarum", "desc": "Hiperemesis Gravidarum Ringan"}
        ],
        "medications": [
            {"keywords": ["asam folat", "folic acid"],
             "name": "Asam Folat 1 mg Tablet", "kfa": "93005501", "dose": "1 tablet (1 mg)", "freq": "1 kali sehari", "route": "Oral", "days": 30, "inst": "Diminum teratur 1 kali sehari."},
            {"keywords": ["kalsium", "calcium", "calcichew"],
             "name": "Kalsium Laktat 500 mg Tablet", "kfa": "93005502", "dose": "1 tablet (500 mg)", "freq": "1 kali sehari", "route": "Oral", "days": 30, "inst": "Diminum pagi hari sesudah sarapan."},
            {"keywords": ["zat besi", "tambah darah", "sulfas ferosus"],
             "name": "Tablet Tambah Darah (Sulfas Ferosus 200 mg)", "kfa": "93005503", "dose": "1 tablet", "freq": "1 kali sehari malam", "route": "Oral", "days": 30, "inst": "Diminum malam hari dengan air putih / jus jeruk (hindari teh/kopi)."}
        ],
        "procedures": [
            {"keywords": ["usg kandungan", "usg janin", "ultrasound obstetri"],
             "code": "88.78", "display": "Diagnostic ultrasound of gravid uterus", "desc": "USG Kandungan 2D/4D & Denyut Jantung Janin (DJJ)"},
            {"keywords": ["pemeriksaan panggul", "inspekulo", "ginekologi"],
             "code": "89.26", "display": "Gynecological examination", "desc": "Pemeriksaan Fisik Obstetri & Palpasi Leopold"}
        ]
    },

    # 6. Layanan Primer & Keluhan Komprehensif (Umum / Puskesmas / Poliklinik)
    "umum": {
        "specialty": "Dokter Umum & Keluhan Komprehensif (Primary Care)",
        "diagnoses": [
            {"keywords": ["pegel linu", "pegal linu", "linu", "kemeng", "linu-linu", "nyeri otot"],
             "code": "M79.1", "display": "Myalgia", "desc": "Mialgia / Pegal Linu / Nyeri Otot"},
            {"keywords": ["kesemutan", "baal", "kebas", "gringgingen", "parestesi", "parestesia"],
             "code": "R20.2", "display": "Paresthesia of skin", "desc": "Parestesia / Kesemutan / Baal"},
            {"keywords": ["mriang", "meriang", "sumeng", "panas dingin", "gerah dingin", "demam"],
             "code": "R50.9", "display": "Fever, unspecified", "desc": "Febris / Meriang / Demam"},
            {"keywords": ["masuk angin", "kembung", "begah", "sebah"],
             "code": "K30", "display": "Dyspepsia", "desc": "Dispepsia / Masuk Angin / Perut Begah"},
            {"keywords": ["nyut-nyutan", "cekot-cekot", "sakit kepala berdenyut"],
             "code": "R51", "display": "Headache", "desc": "Sefalgia / Sakit Kepala Berdenyut"},
            {"keywords": ["puyeng", "kleyengan", "mumet", "pusing berputar", "vertigo"],
             "code": "R42", "display": "Dizziness and giddiness", "desc": "Vertigo / Pusing Berputar / Kleyengan"},
            {"keywords": ["sesek", "sesak napas", "menggeh-menggeh"],
             "code": "R06.0", "display": "Dyspnoea", "desc": "Dispnea / Sesak Napas"},
            {"keywords": ["mencret", "murus", "buang air cair"],
             "code": "A09", "display": "Infectious gastroenteritis and colitis, unspecified", "desc": "Gastroenteritis Akut / Diare / Mencret"},
            {"keywords": ["gatal biduran", "kaligata", "bentol"],
             "code": "L50.9", "display": "Urticaria, unspecified", "desc": "Urtikaria / Biduran / Kaligata"},
            {"keywords": ["asam lambung", "gerd", "ulu hati"],
             "code": "K21.9", "display": "Gastro-esophageal reflux disease without esophagitis", "desc": "GERD / Asam Lambung"}
        ],
        "medications": [
            {"keywords": ["paracetamol", "parasetamol", "panadol", "sanmol"],
             "name": "Paracetamol 500 mg Tablet", "kfa": "93006601", "dose": "1 tablet (500 mg)", "freq": "3-4 kali sehari prn demam", "route": "Oral", "days": 5, "inst": "Diminum bila demam atau nyeri."},
            {"keywords": ["ibuprofen", "proris"],
             "name": "Ibuprofen 400 mg Tablet", "kfa": "93006602", "dose": "1 tablet (400 mg)", "freq": "3 kali sehari", "route": "Oral", "days": 5, "inst": "Diminum sesudah makan untuk redakan nyeri/linu."},
            {"keywords": ["cetirizine", "setirizin", "antihistamin"],
             "name": "Cetirizine 10 mg Tablet", "kfa": "93006603", "dose": "1 tablet (10 mg)", "freq": "1 kali sehari malam", "route": "Oral", "days": 5, "inst": "Diminum malam hari untuk redakan gatal/biduran."},
            {"keywords": ["domperidone", "domperidon", "anti mual"],
             "name": "Domperidone 10 mg Tablet", "kfa": "93006604", "dose": "1 tablet (10 mg)", "freq": "3 kali sehari", "route": "Oral", "days": 5, "inst": "Diminum 15-30 menit sebelum makan."}
        ],
        "procedures": [
            {"keywords": ["pemeriksaan fisik umum", "tanda vital", "inspeksi"],
             "code": "89.7", "display": "General medical examination", "desc": "Pemeriksaan Fisik Diagnostik Umum"},
            {"keywords": ["auskultasi paru", "auskultasi jantung", "pemeriksaan stetoskop"],
             "code": "89.37", "display": "Vital signs measurement", "desc": "Auskultasi Paru & Jantung"}
        ]
    }
}


# ---------------------------------------------------------------------------
# Sovereign Local Medical LLM Engine
# ---------------------------------------------------------------------------

class SovereignLocalMedicalLLM:
    """
    On-premises, privacy-first Sovereign Medical Intelligence Engine.
    Conforms to Permenkes No. 24/2022 and UU PDP No. 27/2022.

    Zero data leaves the hospital perimeter.
    100% Deterministic extraction + Grounded Evidence linking + SATUSEHAT FHIR R4.
    """

    def __init__(self, local_llm_backend: Optional[str] = "sovereign_rule_engine"):
        self.backend = local_llm_backend
        self.ontology = CLINICAL_ONTOLOGY

    def detect_specialty(self, text: str) -> str:
        """Determines the clinical specialty based on transcript vocabulary."""
        text_lower = text.lower()
        scores = {}
        for spec_key, spec_data in self.ontology.items():
            score = 0
            # Check diagnoses keywords
            for diag in spec_data["diagnoses"]:
                for kw in diag["keywords"]:
                    if kw in text_lower:
                        score += 3
            # Check medications keywords
            for med in spec_data["medications"]:
                for kw in med["keywords"]:
                    if kw in text_lower:
                        score += 2
            # Check procedures keywords
            for proc in spec_data["procedures"]:
                for kw in proc["keywords"]:
                    if kw in text_lower:
                        score += 2
            scores[spec_key] = score

        best_spec = max(scores, key=scores.get)
        if scores[best_spec] == 0:
            return "mata"  # default fallback
        return best_spec

    def extract_vitals(self, text: str) -> VitalSigns:
        """Parses vital signs and clinical measurements with standard LOINC codes."""
        from clinical_safety import extract_supported_vitals
        v = extract_supported_vitals(text)
        sys_, dia = v.get("systolic"), v.get("diastolic")
        return VitalSigns(blood_pressure=f"{sys_}/{dia} mmHg" if sys_ and dia else None,
                          systolic_bp=sys_, diastolic_bp=dia, heart_rate=v.get("heart_rate"),
                          respiratory_rate=v.get("respiratory_rate"), temperature=v.get("temperature"),
                          spo2=v.get("spo2"))

    def generate_linked_evidence(self, turns: List[DialogueTurn], keywords: List[str]) -> Optional[LinkedEvidence]:
        """Finds matching audio timestamps and dialogue turns to substantiate clinical claims."""
        for turn in turns:
            turn_text_lower = turn.text.lower()
            for kw in keywords:
                if kw in turn_text_lower:
                    return LinkedEvidence(
                        claim_text=f"Bukti percakapan: \"{turn.text}\"",
                        source_speaker=turn.speaker,
                        audio_start_ms=int(turn.start * 1000),
                        audio_end_ms=int(turn.end * 1000),
                        confidence=0.96,
                        quote_snippet=turn.text
                    )
        if turns:
            first_turn = turns[0]
            return LinkedEvidence(
                claim_text=f"Konteks konsultasi: \"{first_turn.text}\"",
                source_speaker=first_turn.speaker,
                audio_start_ms=int(first_turn.start * 1000),
                audio_end_ms=int(first_turn.end * 1000),
                confidence=0.88,
                quote_snippet=first_turn.text
            )
        return None

    def analyze_consultation(
        self,
        turns: List[DialogueTurn],
        patient_name: str = "Bpk. Rahmat Santoso",
        patient_id: str = "P12345678901",
        doctor_name: str = "dr. Spesialis, Sp.A",
        encounter_id: str = "ENC-RME-20261031-001"
    ) -> SOAPRecord:
        """
        Executes end-to-end clinical analysis:
        1. Segregates subjective complaints and objective observations
        2. Maps ICD-10 diagnoses & ICD-9-CM procedures
        3. Formulates medication prescription with KFA Kemenkes codes
        4. Attaches precise audio millisecond timestamps for Linked Evidence
        """
        combined_text = " ".join([t.text for t in turns])
        specialty_key = self.detect_specialty(combined_text)
        spec_data = self.ontology[specialty_key]

        # Extract vitals
        vitals = self.extract_vitals(combined_text)

        # Extract Diagnoses with ICD-10 & Linked Evidence
        diagnoses: List[DiagnosisItem] = []
        patient_turns = [t for t in turns if t.speaker == "PASIEN"]
        doctor_turns = [t for t in turns if t.speaker == "DOKTER"]

        for idx, d_candidate in enumerate(spec_data["diagnoses"]):
            # Check if any keyword matches
            matched = any(kw in combined_text.lower() for kw in d_candidate["keywords"])
            if matched or (idx == 0 and len(diagnoses) == 0):
                evidence = self.generate_linked_evidence(turns, d_candidate["keywords"])
                item = DiagnosisItem(
                    description=d_candidate["desc"],
                    icd10_code=d_candidate["code"],
                    icd10_display=d_candidate["display"],
                    is_primary=(len(diagnoses) == 0),
                    evidence=evidence
                )
                diagnoses.append(item)
                if len(diagnoses) >= 2:
                    break

        if not diagnoses:
            # Fallback primary diagnosis
            d_fallback = spec_data["diagnoses"][0]
            diagnoses.append(DiagnosisItem(
                description=d_fallback["desc"],
                icd10_code=d_fallback["code"],
                icd10_display=d_fallback["display"],
                is_primary=True,
                evidence=self.generate_linked_evidence(turns, d_fallback["keywords"])
            ))

        # Extract Medications with KFA codes & Linked Evidence
        medications: List[MedicationItem] = []
        for m_candidate in spec_data["medications"]:
            matched = any(kw in combined_text.lower() for kw in m_candidate["keywords"])
            if matched or len(medications) == 0:
                evidence = self.generate_linked_evidence(doctor_turns or turns, m_candidate["keywords"])
                med_item = MedicationItem(
                    drug_name=m_candidate["name"],
                    kfa_code=m_candidate["kfa"],
                    dosage=m_candidate["dose"],
                    frequency=m_candidate["freq"],
                    route=m_candidate["route"],
                    duration_days=m_candidate["days"],
                    instructions=m_candidate["inst"],
                    evidence=evidence
                )
                medications.append(med_item)
                if len(medications) >= 2:
                    break

        # Extract Procedures with ICD-9-CM
        procedures: List[ProcedureItem] = []
        for p_candidate in spec_data["procedures"]:
            matched = any(kw in combined_text.lower() for kw in p_candidate["keywords"])
            if matched or len(procedures) == 0:
                evidence = self.generate_linked_evidence(turns, p_candidate["keywords"])
                proc_item = ProcedureItem(
                    procedure_name=p_candidate["desc"],
                    icd9cm_code=p_candidate["code"],
                    icd9cm_display=p_candidate["display"],
                    evidence=evidence
                )
                procedures.append(proc_item)
                if len(procedures) >= 2:
                    break

        # Extract Subjective details
        chief_complaint = "Keluhan utama dilaporkan pasien saat konsultasi."
        if patient_turns:
            chief_complaint = patient_turns[0].text
        elif turns:
            chief_complaint = turns[0].text

        subjective_narrative = (
            f"Pasien datang dengan keluhan: {chief_complaint}. "
            f"Berdasarkan anamnesis terarah, riwayat gejala telah dirasakan pasien dan mengganggu aktivitas harian. "
            f"Riwayat alergi obat/makanan disangkal atau dalam batas toleransi. Riwayat penyakit dahulu dievaluasi."
        )

        # Objective narrative & physical examination
        objective_narrative = (
            f"Keadaan Umum: Tampak sakit ringan-sedang, Compos Mentis (GCS 15). "
            f"Tanda Vital: Tekanan Darah {vitals.blood_pressure}, Denyut Nadi {vitals.heart_rate} x/menit, "
            f"Frekuensi Pernapasan {vitals.respiratory_rate} x/menit, Suhu {vitals.temperature}°C, SpO2 {vitals.spo2}%."
        )
        physical_exam = (
            f"Pemeriksaan Terarah ({spec_data['specialty']}): "
            f"{procedures[0].procedure_name if procedures else 'Pemeriksaan fisik sesuai panduan klinis'}. "
            f"Ditemukan temuan klinis relevan yang mendukung tegaknya diagnosis kerja."
        )

        assessment_narrative = (
            f"Diagnosis Kerja: [{diagnoses[0].icd10_code}] {diagnoses[0].icd10_display} - {diagnoses[0].description}. "
            + (f"Diagnosis Banding: [{diagnoses[1].icd10_code}] {diagnoses[1].icd10_display}." if len(diagnoses) > 1 else "")
        )

        plan_narrative = (
            f"Diberikan farmakoterapi lini pertama sesuai formularium KFA Kemenkes. "
            f"Edukasi kepatuhan minum obat, istirahat cukup, dan tanda bahaya kegawatan klinis."
        )

        patient_education = (
            "1. Minum/gunakan obat secara teratur sesuai petunjuk dokter dan selesaikan durasi terapi.\n"
            "2. Segera kembali ke fasilitas kesehatan bila muncul keluhan perburukan atau tanda bahaya.\n"
            "3. Jaga kebersihan dan istirahat yang cukup di rumah."
        )

        follow_up_plan = "Kontrol ulang 3 sampai 5 hari kemudian atau bila gejala belum membaik."

        soap_record = SOAPRecord(
            patient_id=patient_id,
            patient_name=patient_name,
            doctor_name=doctor_name,
            encounter_id=encounter_id,
            specialty=spec_data["specialty"],
            chief_complaint=chief_complaint,
            history_of_present_illness=f"Gejala dialami beberapa hari terakhir. {chief_complaint}",
            past_medical_history="Riwayat hipertensi/DM disesuaikan dengan rekam medis terdahulu.",
            allergies="Tidak ada riwayat alergi obat yang diketahui (NKDA).",
            subjective_narrative=subjective_narrative,
            vitals=vitals,
            physical_exam=physical_exam,
            objective_narrative=objective_narrative,
            assessment_narrative=assessment_narrative,
            diagnoses=diagnoses,
            plan_narrative=plan_narrative,
            medications=medications,
            procedures=procedures,
            patient_education=patient_education,
            follow_up_plan=follow_up_plan
        )

        return soap_record

    def generate_satusehat_bundle(self, soap: SOAPRecord, org_id: str = "ORG-RSUD-001") -> Dict[str, Any]:
        """
        Builds standard SATUSEHAT HL7 FHIR R4 Transaction Bundle JSON
        conforming to Kemenkes Permenkes No. 24/2022.
        """
        timestamp = time.strftime("%Y-%m-%dT%H:%M:%S+07:00")
        bundle_id = f"bundle-satusehat-{soap.encounter_id}"

        # 1. Encounter
        encounter_entry = {
            "fullUrl": f"urn:uuid:encounter-{soap.encounter_id}",
            "resource": {
                "resourceType": "Encounter",
                "id": soap.encounter_id,
                "identifier": [{
                    "system": f"http://sys-ids.kemkes.go.id/encounter/{org_id}",
                    "value": soap.encounter_id
                }],
                "status": "finished",
                "class": {
                    "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
                    "code": "AMB",
                    "display": "ambulatory"
                },
                "subject": {
                    "reference": f"Patient/{soap.patient_id}",
                    "display": soap.patient_name
                },
                "participant": [{
                    "type": [{
                        "coding": [{
                            "system": "http://terminology.hl7.org/CodeSystem/v3-ParticipationType",
                            "code": "ATND",
                            "display": "attender"
                        }]
                    }],
                    "individual": {
                        "reference": f"Practitioner/{soap.doctor_name.replace(' ', '_')}",
                        "display": soap.doctor_name
                    }
                }],
                "period": {
                    "start": timestamp,
                    "end": timestamp
                },
                "serviceProvider": {
                    "reference": f"Organization/{org_id}"
                }
            },
            "request": {
                "method": "POST",
                "url": "Encounter"
            }
        }

        entries = [encounter_entry]

        # 2. Conditions (ICD-10)
        for idx, diag in enumerate(soap.diagnoses):
            cond_id = f"cond-{soap.encounter_id}-{idx+1}"
            cond_entry = {
                "fullUrl": f"urn:uuid:{cond_id}",
                "resource": {
                    "resourceType": "Condition",
                    "id": cond_id,
                    "clinicalStatus": {
                        "coding": [{
                            "system": "http://terminology.hl7.org/CodeSystem/condition-clinical",
                            "code": "active",
                            "display": "Active"
                        }]
                    },
                    "category": [{
                        "coding": [{
                            "system": "http://terminology.hl7.org/CodeSystem/condition-category",
                            "code": "encounter-diagnosis",
                            "display": "Encounter Diagnosis"
                        }]
                    }],
                    "code": {
                        "coding": [{
                            "system": "http://hl7.org/fhir/sid/icd-10",
                            "code": diag.icd10_code,
                            "display": diag.icd10_display
                        }],
                        "text": diag.description
                    },
                    "subject": {
                        "reference": f"Patient/{soap.patient_id}",
                        "display": soap.patient_name
                    },
                    "encounter": {
                        "reference": f"Encounter/{soap.encounter_id}"
                    },
                    "recordedDate": timestamp
                },
                "request": {
                    "method": "POST",
                    "url": "Condition"
                }
            }
            entries.append(cond_entry)

        # 3. Observation - Blood Pressure (LOINC 85354-9)
        obs_bp_entry = {
            "fullUrl": f"urn:uuid:obs-bp-{soap.encounter_id}",
            "resource": {
                "resourceType": "Observation",
                "id": f"obs-bp-{soap.encounter_id}",
                "status": "final",
                "category": [{
                    "coding": [{
                        "system": "http://terminology.hl7.org/CodeSystem/observation-category",
                        "code": "vital-signs",
                        "display": "Vital Signs"
                    }]
                }],
                "code": {
                    "coding": [{
                        "system": "http://loinc.org",
                        "code": "85354-9",
                        "display": "Blood pressure panel with all children optional"
                    }]
                },
                "subject": {
                    "reference": f"Patient/{soap.patient_id}",
                    "display": soap.patient_name
                },
                "encounter": {
                    "reference": f"Encounter/{soap.encounter_id}"
                },
                "effectiveDateTime": timestamp,
                "component": [
                    {
                        "code": {
                            "coding": [{
                                "system": "http://loinc.org",
                                "code": "8480-6",
                                "display": "Systolic blood pressure"
                            }]
                        },
                        "valueQuantity": {
                            "value": soap.vitals.systolic_bp,
                            "unit": "mm[Hg]",
                            "system": "http://unitsofmeasure.org",
                            "code": "mm[Hg]"
                        }
                    },
                    {
                        "code": {
                            "coding": [{
                                "system": "http://loinc.org",
                                "code": "8462-4",
                                "display": "Diastolic blood pressure"
                            }]
                        },
                        "valueQuantity": {
                            "value": soap.vitals.diastolic_bp,
                            "unit": "mm[Hg]",
                            "system": "http://unitsofmeasure.org",
                            "code": "mm[Hg]"
                        }
                    }
                ]
            },
            "request": {
                "method": "POST",
                "url": "Observation"
            }
        }
        entries.append(obs_bp_entry)

        # 4. Observation - Heart Rate (LOINC 8867-4)
        obs_hr_entry = {
            "fullUrl": f"urn:uuid:obs-hr-{soap.encounter_id}",
            "resource": {
                "resourceType": "Observation",
                "id": f"obs-hr-{soap.encounter_id}",
                "status": "final",
                "category": [{
                    "coding": [{
                        "system": "http://terminology.hl7.org/CodeSystem/observation-category",
                        "code": "vital-signs"
                    }]
                }],
                "code": {
                    "coding": [{
                        "system": "http://loinc.org",
                        "code": "8867-4",
                        "display": "Heart rate"
                    }]
                },
                "subject": {"reference": f"Patient/{soap.patient_id}"},
                "encounter": {"reference": f"Encounter/{soap.encounter_id}"},
                "effectiveDateTime": timestamp,
                "valueQuantity": {
                    "value": soap.vitals.heart_rate,
                    "unit": "beats/minute",
                    "system": "http://unitsofmeasure.org",
                    "code": "/min"
                }
            },
            "request": {"method": "POST", "url": "Observation"}
        }
        entries.append(obs_hr_entry)

        # 5. MedicationRequest (KFA Kemenkes)
        for idx, med in enumerate(soap.medications):
            med_id = f"medreq-{soap.encounter_id}-{idx+1}"
            med_entry = {
                "fullUrl": f"urn:uuid:{med_id}",
                "resource": {
                    "resourceType": "MedicationRequest",
                    "id": med_id,
                    "status": "active",
                    "intent": "order",
                    "medicationCodeableConcept": {
                        "coding": [{
                            "system": "http://sys-ids.kemkes.go.id/kfa",
                            "code": med.kfa_code,
                            "display": med.drug_name
                        }],
                        "text": med.drug_name
                    },
                    "subject": {
                        "reference": f"Patient/{soap.patient_id}",
                        "display": soap.patient_name
                    },
                    "encounter": {
                        "reference": f"Encounter/{soap.encounter_id}"
                    },
                    "authoredOn": timestamp,
                    "requester": {
                        "reference": f"Practitioner/{soap.doctor_name.replace(' ', '_')}",
                        "display": soap.doctor_name
                    },
                    "dosageInstruction": [{
                        "text": f"{med.dosage}, {med.frequency}, rute {med.route}. {med.instructions}",
                        "route": {
                            "coding": [{
                                "system": "http://standardterms.edqm.eu",
                                "display": med.route
                            }]
                        }
                    }]
                },
                "request": {
                    "method": "POST",
                    "url": "MedicationRequest"
                }
            }
            entries.append(med_entry)

        # 6. Procedures (ICD-9-CM)
        for idx, proc in enumerate(soap.procedures):
            proc_id = f"proc-{soap.encounter_id}-{idx+1}"
            proc_entry = {
                "fullUrl": f"urn:uuid:{proc_id}",
                "resource": {
                    "resourceType": "Procedure",
                    "id": proc_id,
                    "status": "completed",
                    "code": {
                        "coding": [{
                            "system": "http://hl7.org/fhir/sid/icd-9-cm",
                            "code": proc.icd9cm_code,
                            "display": proc.icd9cm_display
                        }],
                        "text": proc.procedure_name
                    },
                    "subject": {
                        "reference": f"Patient/{soap.patient_id}",
                        "display": soap.patient_name
                    },
                    "encounter": {
                        "reference": f"Encounter/{soap.encounter_id}"
                    },
                    "performedDateTime": timestamp
                },
                "request": {
                    "method": "POST",
                    "url": "Procedure"
                }
            }
            entries.append(proc_entry)

        # Build FHIR Transaction Bundle
        bundle = {
            "resourceType": "Bundle",
            "id": bundle_id,
            "type": "transaction",
            "timestamp": timestamp,
            "meta": {
                "profile": [
                    "https://fhir.kemkes.go.id/r4/StructureDefinition/Bundle"
                ]
            },
            "entry": entries
        }

        return bundle


if __name__ == "__main__":
    print("Testing Sovereign Local Medical LLM Engine...")
    llm = SovereignLocalMedicalLLM()

    # Sample dialogue turns
    test_turns = [
        DialogueTurn(1, "DOKTER", 0.0, 2.5, "Selamat pagi Pak, keluhannya apa ya pada matanya?"),
        DialogueTurn(2, "PASIEN", 2.8, 6.5, "Pagi Dok, mata kanan saya merah sekali dan berair terus sejak 3 hari ini."),
        DialogueTurn(3, "DOKTER", 6.8, 10.2, "Apakah ada rasa mengganjal dan silau? Tensi bapak 125/80 mmHg."),
        DialogueTurn(4, "PASIEN", 10.5, 14.0, "Iya Dok, mengganjal seperti kelilipan pasir dan agak perih silau."),
        DialogueTurn(5, "DOKTER", 14.5, 20.0, "Baik saya periksa dengan slit lamp ya. Ditemukan konjungtivitis akut. Saya resepkan tetes mata kloramfenikol.")
    ]

    soap = llm.analyze_consultation(test_turns)
    print("\n[Extracted SOAP Result]:")
    print(f"Specialty : {soap.specialty}")
    print(f"Chief     : {soap.chief_complaint}")
    print(f"Vitals    : {soap.vitals.blood_pressure}")
    print(f"Diagnosis : [{soap.diagnoses[0].icd10_code}] {soap.diagnoses[0].description}")
    print(f"Medication: {soap.medications[0].drug_name} (KFA: {soap.medications[0].kfa_code})")
    print(f"Evidence  : {soap.diagnoses[0].evidence.claim_text} ({soap.diagnoses[0].evidence.audio_start_ms}ms)")

    bundle = llm.generate_satusehat_bundle(soap)
    print(f"\nSATUSEHAT FHIR R4 Bundle Created with {len(bundle['entry'])} entries. ResourceTypes:")
    for e in bundle["entry"]:
        print(f"  * {e['resource']['resourceType']} (ID: {e['resource']['id']})")
    print("\nUnit Test Passed Successfully!")

In [ ]:
%%writefile benchmark_real.py
#!/usr/bin/env python3
"""
Real benchmark toolkit for the Voice-to-RME stack.

Why this exists: the earlier benchmark used a chunk-level test split (every test video also
appears in train), 100-600 clips, and 41 hand-picked clinical terms. This toolkit instead:
  * tags every clip as train_clip / heldout_seen_video / heldout_unseen_video (leakage-aware)
  * drops heldout clips whose exact reference text occurs in train
  * scores thousands of clips with bootstrap 95% CIs, S/D/I breakdown, hallucination rate
  * measures clinical-term recall AND false-term rate over the whole ontology vocabulary
  * measures downstream SOAP extraction on multi-minute sessions (ASR-induced degradation,
    JSON validity, grounding, fabricated vitals) without needing gold SOAP annotations
Pure-python scoring functions are unit-tested on CPU (test_benchmark_real.py).
"""

import collections
import csv
import json
import os
import random
import re
import time
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

SAMPLE_RATE = 16000
MAX_CLIP_SEC = 29.5


# ---------------------------------------------------------------------------
# Text normalisation + error counting
# ---------------------------------------------------------------------------

def normalize(text: str) -> str:
    """Lowercase, strip punctuation/hyphens, collapse whitespace (references carry no casing/punctuation)."""
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", (text or "").lower())).strip()


INDONESIAN_CONVERSATIONAL_MAP = {
    r"\bnggak\b": "tidak",
    r"\bgak\b": "tidak",
    r"\bngga\b": "tidak",
    r"\bndak\b": "tidak",
    r"\btak\b": "tidak",
    r"\budah\b": "sudah",
    r"\bdah\b": "sudah",
    r"\bkalo\b": "kalau",
    r"\bklo\b": "kalau",
    r"\bpake\b": "pakai",
    r"\bgimana\b": "bagaimana",
    r"\bbagaimanakah\b": "bagaimana",
    r"\btau\b": "tahu",
    r"\baja\b": "saja",
    r"\bcuman\b": "cuma",
    r"\bmakasih\b": "terima kasih",
    r"\bya\b": "iya",
    r"\bbanget\b": "sekali",
    r"\btrus\b": "terus",
    r"\bknp\b": "kenapa",
    r"\bngantuk\b": "mengantuk",
    r"\blaper\b": "lapar",
    r"\bcapek\b": "lelah",
    # Fillers & hesitations
    r"\b(?:ee|ehm|eh|ah|uh|oh|nah|kan|lah)\b": "",
    # Clinical units & numbers
    r"(\d+)\s*mg\b": r"\1 miligram",
    r"(\d+)\s*ml\b": r"\1 mililiter",
    r"(\d+)\s*g\b": r"\1 gram",
    r"(\d+)\s*kg\b": r"\1 kilogram",
    r"(\d+)\s*x\s*(\d+)": r"\1 kali \2",
}


def normalize_clinical_conversational(text: str) -> str:
    """
    Standardizes spoken Indonesian conversational particles, contractions,
    and clinical measurements to evaluate true semantic accuracy.
    """
    t = normalize(text)
    for pat, rep in INDONESIAN_CONVERSATIONAL_MAP.items():
        t = re.sub(pat, rep, t)
    return re.sub(r"\s+", " ", t).strip()


def levenshtein(a: Sequence, b: Sequence) -> int:
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def word_error_counts(ref_words: List[str], hyp_words: List[str]) -> Tuple[int, int, int]:
    """Return (substitutions, deletions, insertions) via full DP + backtrace."""
    n, m = len(ref_words), len(hyp_words)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1,
                          d[i - 1][j - 1] + (ref_words[i - 1] != hyp_words[j - 1]))
    i, j, s, dele, ins = n, m, 0, 0, 0
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (ref_words[i - 1] != hyp_words[j - 1]):
            s += ref_words[i - 1] != hyp_words[j - 1]
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            dele += 1
            i -= 1
        else:
            ins += 1
            j -= 1
    return s, dele, ins


def bootstrap_ci(errs: Sequence[int], ns: Sequence[int], B: int = 1000, seed: int = 0) -> Tuple[float, float, float]:
    """Corpus-level rate = sum(errs)/sum(ns), 95% CI by resampling utterances."""
    errs, ns = np.asarray(errs, dtype=float), np.asarray(ns, dtype=float)
    if len(errs) == 0 or ns.sum() == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(errs), size=(B, len(errs)))
    rates = errs[idx].sum(1) / np.maximum(ns[idx].sum(1), 1)
    return float(errs.sum() / ns.sum()), float(np.percentile(rates, 2.5)), float(np.percentile(rates, 97.5))


def wilson_ci(k: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    """95% Wilson interval (%) for a proportion k/n; honest about tiny n."""
    if n <= 0:
        return float("nan"), float("nan")
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / den
    return max(0.0, centre - half) * 100, min(1.0, centre + half) * 100


def mcnemar_exact(b: int, c: int) -> float:
    """Two-sided exact McNemar p-value from discordant counts (b: A-only wins, c: B-only wins)."""
    from math import comb
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)


def collapse_repetitions(text: str, max_ngram: int = 6, keep: int = 2) -> str:
    """Collapse a word n-gram repeated more than `keep` times in a row (decoder loops) down to `keep` copies."""
    w = (text or "").split()
    changed = True
    while changed:
        changed = False
        for n in range(1, max_ngram + 1):
            i = 0
            while i + n * (keep + 1) <= len(w):
                unit = w[i:i + n]
                reps = 1
                while w[i + reps * n:i + (reps + 1) * n] == unit:
                    reps += 1
                if reps > keep:
                    del w[i + keep * n:i + reps * n]
                    changed = True
                i += 1
    return " ".join(w)


def compression_ratio(text: str) -> float:
    """len(utf8) / len(zlib(utf8)); Whisper's own loop heuristic flags > 2.4."""
    import zlib
    b = (text or "").encode("utf-8")
    return len(b) / max(len(zlib.compress(b)), 1) if b else 0.0


def degenerate_reason(text: str, dur: float, max_ratio: float = 2.4, max_wps: float = 6.0) -> Optional[str]:
    """Why a hypothesis looks like a decoder failure (loop / runaway), judged without the reference."""
    words = (text or "").split()
    if not words:
        return None  # silence is a legitimate output
    if len(words) >= 12 and compression_ratio(text) > max_ratio:
        return "COMPRESSION"
    if len(words) >= 12 and len(words) / max(dur, 0.5) > max_wps:
        return "WORDS_PER_SEC"
    if len(words) >= 8 and len(collapse_repetitions(text).split()) < 0.7 * len(words):
        return "REPEATED_NGRAM"
    return None


def clip_error_arrays(items: List[Dict[str, Any]]) -> Tuple[np.ndarray, np.ndarray]:
    """Per-clip (word errors, reference words) using strict normalisation."""
    errs, ns = [], []
    for it in items:
        rw, hw = normalize(it["ref"]).split(), normalize(it["hyp"] or "").split()
        s, d, i = word_error_counts(rw, hw)
        errs.append(s + d + i)
        ns.append(max(len(rw), 1))
    return np.asarray(errs, dtype=float), np.asarray(ns, dtype=float)


def paired_bootstrap(items_a: List[Dict[str, Any]], items_b: List[Dict[str, Any]], B: int = 2000, seed: int = 0) -> Dict[str, Any]:
    """Paired comparison of two systems on the SAME clips (items must be aligned by index).
    delta = WER_a - WER_b (pp). Negative => A better. p is the two-sided bootstrap p-value for delta == 0."""
    if len(items_a) != len(items_b):
        raise ValueError("paired_bootstrap needs aligned item lists")
    for a, b in zip(items_a, items_b):
        if a["ref"] != b["ref"] or a.get("chunk_id") != b.get("chunk_id"):
            raise ValueError("paired comparison has mismatched references or clip IDs")
    ea, na = clip_error_arrays(items_a)
    eb, nb = clip_error_arrays(items_b)
    if len(ea) == 0:
        return {"n": 0, "wer_a": float("nan"), "wer_b": float("nan"), "delta_pp": float("nan"),
                "delta_ci95": [float("nan")] * 2, "p_value": float("nan"), "a_better_clip_frac": float("nan")}
    rng = np.random.default_rng(seed)
    group_ids = [a.get("video_id", a.get("chunk_id", i)) for i, a in enumerate(items_a)]
    if any(a.get("video_id") != b.get("video_id") for a, b in zip(items_a, items_b)):
        raise ValueError("paired comparison has mismatched video IDs")
    groups = list(dict.fromkeys(group_ids))
    # Resample whole videos to respect correlation among clips from one recording.
    sums = np.asarray([[ea[jj].sum(), na[jj].sum(), eb[jj].sum(), nb[jj].sum()]
                       for g in groups for jj in [[i for i, x in enumerate(group_ids) if x == g]]])
    idx = rng.integers(0, len(groups), size=(B, len(groups)))
    draws = sums[idx].sum(1)
    deltas = (draws[:, 0] / draws[:, 1] - draws[:, 2] / draws[:, 3]) * 100
    p = 2 * min((deltas >= 0).mean(), (deltas <= 0).mean())
    wa, wb = ea / na, eb / nb
    return {"n_clusters": len(groups), "bootstrap_unit": "video" if all("video_id" in a for a in items_a) else "clip", "n": len(ea), "wer_a": float(ea.sum() / na.sum() * 100), "wer_b": float(eb.sum() / nb.sum() * 100),
            "delta_pp": float((ea.sum() / na.sum() - eb.sum() / nb.sum()) * 100),
            "delta_ci95": [float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))],
            "p_value": float(min(1.0, max(p, 1.0 / B))),
            "a_better_clip_frac": float((wa < wb).mean() * 100), "b_better_clip_frac": float((wb < wa).mean() * 100),
            "median_clip_wer_a": float(np.median(wa) * 100), "median_clip_wer_b": float(np.median(wb) * 100)}


def paired_term_test(items_a: List[Dict[str, Any]], items_b: List[Dict[str, Any]], terms: Sequence[str]) -> Dict[str, Any]:
    """Per-(clip, reference-term) hit/miss for A and B, then exact McNemar on discordant pairs."""
    both = a_only = b_only = neither = 0
    for ia, ib in zip(items_a, items_b):
        pr = f" {normalize(ia['ref'])} "
        ha, hb = f" {normalize(ia['hyp'] or '')} ", f" {normalize(ib['hyp'] or '')} "
        for t in terms:
            if _present(t, pr):
                x, y = _present(t, ha), _present(t, hb)
                both += x and y
                a_only += x and not y
                b_only += y and not x
                neither += (not x) and (not y)
    n = both + a_only + b_only + neither
    return {"term_occurrences": n, "both_hit": both, "a_only": a_only, "b_only": b_only, "neither": neither,
            "p_value": mcnemar_exact(a_only, b_only),
            "enough_data": n >= 100}



# ---------------------------------------------------------------------------
# Hallucination + clinical-term metrics
# ---------------------------------------------------------------------------

def hallucination_flag(ref: str, hyp: str, guardrail=None) -> Optional[str]:
    """Reason string if the hypothesis looks hallucinated relative to the reference, else None."""
    hn, rn = hyp.split(), ref.split()
    if not hn:
        return "EMPTY"
    if len(hn) >= 10 and len(hn) > 2.5 * max(len(rn), 1):
        return "LENGTH_RATIO"
    if guardrail is not None:
        bad, why = guardrail.is_hallucination(hyp)
        if bad and not why.startswith("EMPTY"):
            return why.split(" ")[0]
    return None


def build_term_list(ontology: Dict[str, Any], min_len: int = 4) -> List[str]:
    terms = set()
    for spec in ontology.values():
        for kind in ("diagnoses", "medications", "procedures"):
            for item in spec[kind]:
                for kw in item["keywords"]:
                    t = normalize(kw)
                    if len(t) >= min_len:
                        terms.add(t)
    return sorted(terms)


def _present(term: str, padded_text: str) -> bool:
    return f" {term} " in padded_text


def term_stats(ref: str, hyp: str, terms: Iterable[str]) -> Tuple[int, int, int]:
    """(tp, fn, fp): term in both / only in ref / only in hyp. Inputs must be normalized."""
    pr, ph = f" {ref} ", f" {hyp} "
    tp = fn = fp = 0
    for t in terms:
        a, b = _present(t, pr), _present(t, ph)
        tp += a and b
        fn += a and not b
        fp += b and not a
    return tp, fn, fp


def term_groups(ontology: Dict[str, Any], min_len: int = 4) -> Dict[str, str]:
    """normalised term -> 'specialty/kind' (e.g. 'mata/medications'); first occurrence wins."""
    out = {}
    for spec_name, spec in ontology.items():
        for kind in ("diagnoses", "medications", "procedures"):
            for item in spec[kind]:
                for kw in item["keywords"]:
                    t = normalize(kw)
                    if len(t) >= min_len:
                        out.setdefault(t, f"{spec_name}/{kind}")
    return out


def term_recall_by_group(items: List[Dict[str, Any]], groups: Dict[str, str]) -> Dict[str, Dict[str, Any]]:
    """Per ontology group: reference occurrences, hits, recall (%) and Wilson 95% CI."""
    tp, n = collections.Counter(), collections.Counter()
    for it in items:
        pr, ph = f" {normalize(it['ref'])} ", f" {normalize(it['hyp'] or '')} "
        for t, g in groups.items():
            if _present(t, pr):
                n[g] += 1
                tp[g] += _present(t, ph)
    return {g: {"n": n[g], "hit": tp[g], "recall": 100.0 * tp[g] / n[g], "ci95": list(wilson_ci(tp[g], n[g]))}
            for g in sorted(n)}


def score_items(items: List[Dict[str, Any]], terms: Sequence[str], guardrail=None, B: int = 1000) -> Dict[str, Any]:
    """items: dicts with ref, hyp, dur. Returns corpus metrics with CIs for both strict and clinical normalized."""
    w_err, w_n, c_err, c_n = [], [], [], []
    w_err_clin, w_n_clin, c_err_clin, c_n_clin = [], [], [], []
    S = D = I = 0
    halluc = collections.Counter()
    tp = fn = fp = 0
    hours = 0.0
    for it in items:
        r, h = normalize(it["ref"]), normalize(it["hyp"] or "")
        rw, hw = r.split(), h.split()
        s, d, i = word_error_counts(rw, hw)
        S, D, I = S + s, D + d, I + i
        w_err.append(s + d + i)
        w_n.append(max(len(rw), 1))
        c_err.append(levenshtein(r, h))
        c_n.append(max(len(r), 1))

        # Clinical conversational normalization
        rc = normalize_clinical_conversational(it["ref"])
        hc = normalize_clinical_conversational(it["hyp"] or "")
        rwc, hwc = rc.split(), hc.split()
        sc, dc, ic = word_error_counts(rwc, hwc)
        w_err_clin.append(sc + dc + ic)
        w_n_clin.append(max(len(rwc), 1))
        c_err_clin.append(levenshtein(rc, hc))
        c_n_clin.append(max(len(rc), 1))

        hours += it.get("dur", 0.0) / 3600
        flag = hallucination_flag(r, h, guardrail)
        if flag:
            halluc[flag] += 1
        a, b, c = term_stats(r, h, terms)
        tp, fn, fp = tp + a, fn + b, fp + c
    n = len(items)
    clip_wer = np.asarray(w_err, dtype=float) / np.maximum(np.asarray(w_n, dtype=float), 1)
    collapsed = [dict(it, hyp=collapse_repetitions(it["hyp"] or "")) for it in items]
    ce, cn = clip_error_arrays(collapsed)
    wer, wlo, whi = bootstrap_ci(w_err, w_n, B)
    cer, clo, chi = bootstrap_ci(c_err, c_n, B)
    wer_c, wlo_c, whi_c = bootstrap_ci(w_err_clin, w_n_clin, B)
    cer_c, clo_c, chi_c = bootstrap_ci(c_err_clin, c_n_clin, B)
    N = max(sum(w_n), 1)
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    return {
        "n_clips": n, "hours": round(hours, 3), "ref_words": int(sum(w_n)),
        "wer": wer * 100, "wer_ci95": [wlo * 100, whi * 100],
        "cer": cer * 100, "cer_ci95": [clo * 100, chi * 100],
        "wer_clinical": wer_c * 100, "wer_clinical_ci95": [wlo_c * 100, whi_c * 100],
        "cer_clinical": cer_c * 100, "cer_clinical_ci95": [clo_c * 100, chi_c * 100],
        "wer_clip_median": float(np.median(clip_wer) * 100) if n else float("nan"),
        "wer_clip_p90": float(np.percentile(clip_wer, 90) * 100) if n else float("nan"),
        "clips_wer_over_100_pct": float((clip_wer > 1.0).mean() * 100) if n else float("nan"),
        "wer_loop_collapsed": float(ce.sum() / max(cn.sum(), 1) * 100),
        "clinical_term_recall_ci95": list(wilson_ci(tp, tp + fn)),
        "clinical_term_enough_data": (tp + fn) >= 100,
        "clinical_term_tp_fn_fp": [tp, fn, fp],
        "sub_rate": S / N * 100, "del_rate": D / N * 100, "ins_rate": I / N * 100,
        "hallucination_rate": sum(halluc.values()) / max(n, 1) * 100,
        "hallucination_reasons": dict(halluc),
        "clinical_term_occurrences": tp + fn, "clinical_term_recall": rec * 100,
        "clinical_term_precision": prec * 100,
        "clinical_term_f1": (2 * prec * rec / (prec + rec) * 100) if prec == prec and rec == rec and prec + rec else float("nan"),
    }


# ---------------------------------------------------------------------------
# Manifests, leakage-aware tagging, stratified sampling
# ---------------------------------------------------------------------------

def _video_from_file(audio_file: str) -> str:
    return os.path.splitext(audio_file)[0].split("_")[-1]


def normalize_rows(manifest: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    rows = []
    for x in manifest:
        rows.append({
            "chunk_id": x["chunk_id"], "audio_file": x["audio_file"],
            "dur": float(x.get("duration_seconds", x.get("duration", 0.0))),
            "ref": x["text"],
            "video_id": x.get("video_id") or _video_from_file(x["audio_file"]),
            "level": x.get("source_level") or ("Level_2_Clinical_Education_OSCE" if x["chunk_id"].startswith("CLEAN") else "unknown"),
            "category": x.get("clinical_category") or x.get("topic") or "unknown",
        })
    return rows


def _shingles(text: str, k: int = 4) -> set:
    w = normalize(text).split()
    return {" ".join(w[i:i + k]) for i in range(len(w) - k + 1)}


def tag_rows(train: List[Dict[str, Any]], test: List[Dict[str, Any]], near_dup_threshold: Optional[float] = 0.6,
             stats: Optional[Dict[str, int]] = None) -> List[Dict[str, Any]]:
    """Tag clips relative to the manifest the adapter was trained on.
    Heldout clips are dropped when (a) their normalised text equals a train text, or (b) at least
    `near_dup_threshold` of their word 4-gram shingles occur in train (near-duplicate; None disables).
    Drop counts are written to `stats` if given."""
    train_vids = {r["video_id"] for r in train}
    train_texts = {normalize(r["ref"]) for r in train}
    train_shingles = set()
    if near_dup_threshold is not None:
        for r in train:
            train_shingles |= _shingles(r["ref"])
    out = [dict(r, tag="train_clip") for r in train]
    exact = near = 0
    for r in test:
        if normalize(r["ref"]) in train_texts:
            exact += 1
            continue
        if near_dup_threshold is not None:
            sh = _shingles(r["ref"])
            if sh and len(sh & train_shingles) / len(sh) >= near_dup_threshold:
                near += 1
                continue
        out.append(dict(r, tag="heldout_seen_video" if r["video_id"] in train_vids else "heldout_unseen_video"))
    if stats is not None:
        stats.update(dropped_exact=exact, dropped_near=near)
    return out


def eligible(r: Dict[str, Any], min_words: int = 3) -> bool:
    return 0.5 <= r["dur"] <= MAX_CLIP_SEC and len(normalize(r["ref"]).split()) >= min_words


def stratified_sample(rows: List[Dict[str, Any]], n: int, key: str = "category", seed: int = 0) -> List[Dict[str, Any]]:
    """Proportional allocation per stratum (>=1 each), remainder filled randomly. Deterministic."""
    rng = random.Random(seed)
    if n >= len(rows):
        return list(rows)
    groups = collections.defaultdict(list)
    for r in rows:
        groups[r[key]].append(r)
    for g in groups.values():
        rng.shuffle(g)
    total = len(rows)
    picked = []
    for g in groups.values():
        picked += g[:max(1, round(n * len(g) / total))]
    picked = picked[:n] if len(picked) > n else picked
    if len(picked) < n:
        ids = {r["chunk_id"] for r in picked}
        rest = [r for r in rows if r["chunk_id"] not in ids]
        rng.shuffle(rest)
        picked += rest[:n - len(picked)]
    return picked


def build_pool(train_rows, test_rows, n_per_tag: int, seed: int = 0,
               near_dup_threshold: Optional[float] = 0.6) -> List[Dict[str, Any]]:
    tagged = [r for r in tag_rows(train_rows, test_rows, near_dup_threshold) if eligible(r)]
    pool = []
    for tag in ("train_clip", "heldout_seen_video", "heldout_unseen_video"):
        pool += stratified_sample([r for r in tagged if r["tag"] == tag], n_per_tag, seed=seed)
    return pool


# ---------------------------------------------------------------------------
# Sessions for downstream (SOAP) evaluation
# ---------------------------------------------------------------------------

def _chunk_no(audio_file: str) -> int:
    m = re.search(r"chk_(\d+)_", audio_file)
    return int(m.group(1)) if m else -1


def build_sessions(all_rows: List[Dict[str, Any]], n: int, window: int = 12, seed: int = 0,
                   level_prefix: str = "Level_3") -> List[List[Dict[str, Any]]]:
    """Non-overlapping windows of strictly consecutive chunks from one video (multi-minute 'consultations')."""
    by_vid = collections.defaultdict(list)
    for r in all_rows:
        if r["level"].startswith(level_prefix) and _chunk_no(r["audio_file"]) >= 0 and r["dur"] <= MAX_CLIP_SEC:
            by_vid[r["video_id"]].append(r)
    cands = []
    for rows in by_vid.values():
        rows.sort(key=lambda r: _chunk_no(r["audio_file"]))
        runs, cur = [], [rows[0]]
        for r in rows[1:]:
            if _chunk_no(r["audio_file"]) == _chunk_no(cur[-1]["audio_file"]) + 1:
                cur.append(r)
            else:
                runs.append(cur)
                cur = [r]
        runs.append(cur)
        for run in runs:  # windows only inside strictly consecutive runs
            cands += [run[i:i + window] for i in range(0, len(run) - window + 1, window)]
    random.Random(seed).shuffle(cands)
    out, per_video = [], collections.Counter()
    for w in cands:  # at most 2 sessions per video for diversity
        if per_video[w[0]["video_id"]] >= 2:
            continue
        per_video[w[0]["video_id"]] += 1
        out.append(w)
        if len(out) >= n:
            break
    return out


def vitals_unsupported(soap, transcript: str) -> List[str]:
    """Names of vitals reported in the SOAP whose number never appears in the transcript (possible fabrication)."""
    from clinical_safety import filter_vitals
    v = soap.vitals
    _, bad = filter_vitals(dict(systolic=v.systolic_bp, diastolic=v.diastolic_bp,
                               heart_rate=v.heart_rate, respiratory_rate=v.respiratory_rate,
                               temperature=v.temperature, spo2=v.spo2), transcript)
    return bad


def code_sets(soap) -> Dict[str, set]:
    return {
        "icd10": {d.icd10_code for d in soap.diagnoses if d.icd10_code != "UNMAPPED"},
        "kfa": {m.kfa_code for m in soap.medications if m.kfa_code != "UNMAPPED"},
        "icd9cm": {p.icd9cm_code for p in soap.procedures if p.icd9cm_code != "UNMAPPED"},
    }


def micro_prf(pairs: List[Tuple[set, set]]) -> Dict[str, float]:
    """pairs of (gold_set, pred_set) -> micro precision/recall/F1 (%)."""
    tp = sum(len(g & p) for g, p in pairs)
    fp = sum(len(p - g) for g, p in pairs)
    fn = sum(len(g - p) for g, p in pairs)
    pr = tp / (tp + fp) if tp + fp else float("nan")
    rc = tp / (tp + fn) if tp + fn else float("nan")
    f1 = 2 * pr * rc / (pr + rc) if pr == pr and rc == rc and pr + rc else float("nan")
    return {"precision": pr * 100, "recall": rc * 100, "f1": f1 * 100, "gold_items": tp + fn}


# ---------------------------------------------------------------------------
# GPU runners (exercised in Colab, not unit-tested here)
# ---------------------------------------------------------------------------

class CTranslate2Pipe:
    """High-throughput CTranslate2 INT8/FP16 wrapper for Whisper inference."""
    def __init__(self, model_path: str, device: str = "cuda", compute_type: str = "float16"):
        import ctranslate2
        from transformers import WhisperProcessor
        self.model = ctranslate2.models.Whisper(model_path, device=device, compute_type=compute_type)
        self.proc = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo")

    def __call__(self, inputs: List[Dict[str, Any]], batch_size: int = 16, generate_kwargs: Optional[Dict[str, Any]] = None):
        import ctranslate2
        outs = []
        beams = generate_kwargs.get("num_beams", 1) if generate_kwargs else 1
        for it in inputs:
            feats = self.proc.feature_extractor(it["raw"], sampling_rate=SAMPLE_RATE, return_tensors="np").input_features
            s_view = ctranslate2.StorageView.from_array(feats)
            prompt = self.proc.tokenizer.convert_tokens_to_ids([
                "<|startoftranscript|>", "<|id|>", "<|transcribe|>", "<|notimestamps|>"
            ])
            res = self.model.generate(s_view, [prompt], beam_size=beams)
            text = self.proc.tokenizer.decode(res[0].sequences_ids[0], skip_special_tokens=True)
            outs.append({"text": text.strip()})
        return outs


def build_asr_pipe(adapter_dir: Optional[str] = None, model_id: str = "openai/whisper-large-v3-turbo",
                   backend: str = "transformers", compute_type: str = "float16"):
    import torch, os
    from transformers import pipeline, WhisperForConditionalGeneration, WhisperProcessor

    if backend == "ctranslate2":
        return CTranslate2Pipe(model_path=adapter_dir or model_id, compute_type=compute_type)

    target_id = model_id
    use_peft = False
    if adapter_dir and os.path.isdir(adapter_dir):
        has_adapter_cfg = os.path.exists(os.path.join(adapter_dir, "adapter_config.json"))
        has_model_weights = os.path.exists(os.path.join(adapter_dir, "model.safetensors")) or os.path.exists(os.path.join(adapter_dir, "pytorch_model.bin"))
        if has_model_weights and not has_adapter_cfg:
            # Standalone merged model
            target_id = adapter_dir
        elif has_adapter_cfg:
            use_peft = True
    elif adapter_dir:
        use_peft = True

    proc = WhisperProcessor.from_pretrained(target_id)
    try:
        model = WhisperForConditionalGeneration.from_pretrained(target_id, dtype=torch.float16, low_cpu_mem_usage=True)
    except TypeError:  # transformers < 4.56
        model = WhisperForConditionalGeneration.from_pretrained(target_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
    if use_peft and adapter_dir:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_dir).merge_and_unload()
    # A model saved from a training run can carry use_cache=False (needed for gradient checkpointing); decoding
    # without the KV cache is slower and needs more memory, which would skew speed/VRAM comparisons.
    model.config.use_cache = True
    model.generation_config.forced_decoder_ids = None
    model.to("cuda").eval()
    return pipeline("automatic-speech-recognition", model=model, tokenizer=proc.tokenizer,
                    feature_extractor=proc.feature_extractor, device=0)


def _load(path: str):
    import librosa
    try:
        return librosa.load(path, sr=SAMPLE_RATE)[0]
    except Exception:
        return None


# "plain": decoding exactly as in the original benchmark. "guarded": loop protection applied
# identically to every model, so the baseline is not penalised for decoder repetition loops.
# "beam_fallback": beam search on top of the guard, then every output that degenerate_reason() flags is
# decoded again with FALLBACK_KWARGS and replaced if the retry is no longer degenerate (Whisper-style fallback,
# but driven by observable failure only, so it never sees the reference).
DECODE_PROFILES = {
    "plain": {},
    "guarded": {"no_repeat_ngram_size": 6, "repetition_penalty": 1.05, "max_new_tokens": 200},
    "beam_fallback": {"num_beams": 4, "no_repeat_ngram_size": 6, "repetition_penalty": 1.05,
                      "max_new_tokens": 200, "_fallback": True},
}
FALLBACK_KWARGS = {"num_beams": 5, "no_repeat_ngram_size": 3, "repetition_penalty": 1.2, "max_new_tokens": 160}


def transcribe_rows(pipe, rows: List[Dict[str, Any]], audio_dir: str, batch_size: int = 16,
                    workers: int = 8, beams: int = 1, shard: int = 256,
                    profile: str = "plain") -> Tuple[Dict[str, Optional[str]], Dict[str, float]]:
    """Returns (hyps by chunk_id, timing). Missing/undecodable audio -> None (reported, never silently scored as empty)."""
    import torch
    from concurrent.futures import ThreadPoolExecutor
    from tqdm.auto import tqdm
    hyps: Dict[str, Optional[str]] = {}
    gpu_s = audio_s = 0.0
    warmed = False
    wall0 = time.time()
    gk = {"language": "indonesian", "task": "transcribe", "num_beams": beams, **DECODE_PROFILES[profile]}
    use_fallback = gk.pop("_fallback", False)
    fb_kwargs = {"language": "indonesian", "task": "transcribe", **FALLBACK_KWARGS}
    n_flagged = n_fixed = 0
    with ThreadPoolExecutor(workers) as ex:
        for i in tqdm(range(0, len(rows), shard), desc="ASR"):
            part = rows[i:i + shard]
            auds = list(ex.map(lambda r: _load(os.path.join(audio_dir, r["audio_file"])), part))
            ok = [(r, a) for r, a in zip(part, auds) if a is not None]
            for r, a in zip(part, auds):
                if a is None:
                    hyps[r["chunk_id"]] = None
            if not ok:
                continue
            if not warmed:  # untimed warm-up so CUDA init/kernel compilation is not billed to whichever model runs first
                pipe([{"raw": a, "sampling_rate": SAMPLE_RATE} for _, a in ok[:2]], batch_size=2, generate_kwargs=gk)
                torch.cuda.reset_peak_memory_stats()
                warmed = True
            torch.cuda.synchronize()
            t0 = time.time()
            outs = pipe([{"raw": a, "sampling_rate": SAMPLE_RATE} for _, a in ok],
                        batch_size=batch_size, generate_kwargs=gk)
            texts = [o["text"].strip() for o in outs]
            if use_fallback:
                bad = [k for k, ((_, a), t) in enumerate(zip(ok, texts))
                       if degenerate_reason(t, len(a) / SAMPLE_RATE)]
                if bad:
                    n_flagged += len(bad)
                    retry = pipe([{"raw": ok[k][1], "sampling_rate": SAMPLE_RATE} for k in bad],
                                 batch_size=max(1, batch_size // 2), generate_kwargs=fb_kwargs)
                    for k, o in zip(bad, retry):
                        t = o["text"].strip()
                        if not degenerate_reason(t, len(ok[k][1]) / SAMPLE_RATE):
                            texts[k] = t
                            n_fixed += 1
            torch.cuda.synchronize()
            gpu_s += time.time() - t0
            for (r, a), t in zip(ok, texts):
                hyps[r["chunk_id"]] = t
                audio_s += len(a) / SAMPLE_RATE
    wall = time.time() - wall0
    return hyps, {"gpu_s": gpu_s, "wall_s": wall, "audio_s": audio_s,
                  "rtf_gpu": gpu_s / max(audio_s, 1e-9), "rtf_wall": wall / max(audio_s, 1e-9),
                  "x_realtime_gpu": audio_s / max(gpu_s, 1e-9),
                  "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
                  "missing_audio": sum(v is None for v in hyps.values()),
                  "fallback_flagged": n_flagged, "fallback_fixed": n_fixed}


def export_verification_sheet(path: str, rows: List[Dict[str, Any]], hyps: Dict[str, Dict[str, str]]) -> None:
    """CSV for clinicians: fill corrected_text with the true transcript. Scoring against it gives the real WER."""
    variants = sorted(hyps)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["chunk_id", "audio_file", "current_reference"] + [f"hyp_{v}" for v in variants] + ["corrected_text"])
        for r in rows:
            w.writerow([r["chunk_id"], r["audio_file"], r["ref"]] + [hyps[v].get(r["chunk_id"], "") for v in variants] + [""])


def load_verified(path: str) -> List[Dict[str, Any]]:
    out = []
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if (r.get("corrected_text") or "").strip():
                out.append({"chunk_id": r["chunk_id"], "ref": r["corrected_text"].strip(), "audio_file": r["audio_file"]})
    return out

In [ ]:
%%writefile rme_asr_v2.py
#!/usr/bin/env python3
"""
Voice-to-RME ASR v2: data curation, training recipe and robust decoding for Whisper large-v3-turbo.

What changed vs v1 (whisper_turbo_rme_enterprise_a100_adapter: LoRA r=32 on q_proj/v_proj, 3 epochs) and why:
  * split by VIDEO into train/dev/test. In v1 every test video was also in train, so generalisation to a new
    doctor/room/microphone was never measured (audit finding #2).
  * label curation. The references are YouTube auto-captions. Dropped from training: clips >30 s (Whisper only
    sees 30 s, so the tail of the label teaches it to invent text), implausibly slow labels (<0.6 words/s:
    caption missing most of the speech, teaches deletions), implausibly fast labels and looping captions
    ("hai hai hai hai").
  * LoRA on every attention and MLP projection of encoder and decoder (v1: q_proj/v_proj only, ~0.4% params).
  * SpecAugment + waveform augmentation (gain, noise at random SNR, speed perturbation) against overfitting to
    the recording conditions of a few hundred videos.
  * checkpoint selection by WER on unseen dev videos (v1 had no WER-based selection).
  * decoding: anti-loop guard plus targeted re-decode of degenerate outputs (see benchmark_real.degenerate_reason).
  * export: merged fp16 weights with use_cache=True and Indonesian/transcribe generation defaults.
Pure-python parts are unit-tested on CPU (test_rme_asr_v2.py); GPU parts are exercised in Colab.
"""

import collections
import hashlib
import os
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np

from benchmark_real import SAMPLE_RATE, collapse_repetitions, normalize, word_error_counts

BASE_MODEL_ID = "openai/whisper-large-v3-turbo"
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
MAX_AUDIO_SEC = 30.0
MAX_LABEL_TOKENS = 448


# ---------------------------------------------------------------------------
# Video-level split
# ---------------------------------------------------------------------------

def _rank(seed: int, video_id: str) -> str:
    return hashlib.md5(f"{seed}:{video_id}".encode()).hexdigest()


def split_videos(rows: Sequence[Dict[str, Any]], dev_frac: float = 0.05, test_frac: float = 0.10,
                 seed: int = 13) -> Dict[str, str]:
    """video_id -> "train" | "dev" | "test". Whole videos only, so no speaker/room leaks across splits.
    Stratified by each video's majority level; fractions are measured in clips, not videos, so one very long
    video cannot silently become most of the test set. Deterministic for a given seed and manifest."""
    clips = collections.Counter(r["video_id"] for r in rows)
    levels = collections.defaultdict(collections.Counter)
    for r in rows:
        levels[r["video_id"]][r["level"]] += 1
    by_level = collections.defaultdict(list)
    for v, c in levels.items():
        by_level[c.most_common(1)[0][0]].append(v)
    out = {}
    for vids in by_level.values():
        vids.sort(key=lambda v: _rank(seed, v))
        total = sum(clips[v] for v in vids)
        test_n = dev_n = 0
        for v in vids:
            if test_n < test_frac * total:
                out[v], test_n = "test", test_n + clips[v]
            elif dev_n < dev_frac * total:
                out[v], dev_n = "dev", dev_n + clips[v]
            else:
                out[v] = "train"
    return out


# ---------------------------------------------------------------------------
# Label curation
# ---------------------------------------------------------------------------

def label_issue(row: Dict[str, Any], min_dur: float = 1.0, max_dur: float = MAX_AUDIO_SEC,
                min_wps: float = 0.6, max_wps: float = 4.5, min_words: int = 2) -> Optional[str]:
    """Reason a training clip should be dropped, or None if it looks usable."""
    words = normalize(row["ref"]).split()
    dur = float(row["dur"])
    if len(words) < min_words:
        return "TOO_FEW_WORDS"
    if dur > max_dur:
        return "TOO_LONG"
    if dur < min_dur:
        return "TOO_SHORT"
    wps = len(words) / dur
    if wps < min_wps:
        return "SLOW_LABEL"
    if wps > max_wps:
        return "FAST_LABEL"
    if len(collapse_repetitions(" ".join(words)).split()) < 0.85 * len(words):
        return "LOOPING_LABEL"
    return None


def curate(rows: Sequence[Dict[str, Any]], **kw) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Split rows into kept clips and a Counter of drop reasons."""
    kept, dropped = [], collections.Counter()
    for r in rows:
        why = label_issue(r, **kw)
        if why:
            dropped[why] += 1
        else:
            kept.append(r)
    return kept, dict(dropped)


# ---------------------------------------------------------------------------
# Waveform augmentation (numpy only, runs inside DataLoader workers)
# ---------------------------------------------------------------------------

@dataclass
class WaveAugment:
    p_speed: float = 0.3
    speed_range: tuple = (0.9, 1.1)
    p_noise: float = 0.4
    snr_db: tuple = (10.0, 30.0)
    p_gain: float = 0.5
    gain_db: tuple = (-6.0, 6.0)

    def __call__(self, audio: np.ndarray, rng: np.random.Generator) -> np.ndarray:
        x = np.asarray(audio, dtype=np.float32)
        if len(x) == 0:
            return x
        if rng.random() < self.p_speed:
            speed = rng.uniform(*self.speed_range)
            n = int(round(len(x) / speed))
            if n / SAMPLE_RATE <= MAX_AUDIO_SEC:  # never push a clip past Whisper's window
                x = np.interp(np.linspace(0, len(x) - 1, n), np.arange(len(x)), x).astype(np.float32)
        if rng.random() < self.p_noise:
            white = rng.standard_normal(len(x)).astype(np.float32)
            brown = np.cumsum(white)
            brown -= np.linspace(brown[0], brown[-1], len(brown))  # remove drift
            noise = white if rng.random() < 0.5 else brown
            sig_p = float(np.mean(x ** 2)) + 1e-10
            noise_p = float(np.mean(noise ** 2)) + 1e-10
            snr = rng.uniform(*self.snr_db)
            x = x + noise * np.sqrt(sig_p / (noise_p * 10 ** (snr / 10)))
        if rng.random() < self.p_gain:
            x = x * 10 ** (rng.uniform(*self.gain_db) / 20)
        return np.clip(x, -1.0, 1.0).astype(np.float32)


# ---------------------------------------------------------------------------
# Training data plumbing
# ---------------------------------------------------------------------------

def load_audio(path: str) -> np.ndarray:
    import librosa
    return librosa.load(path, sr=SAMPLE_RATE)[0]


class ClipDataset:
    """List-backed dataset of {path, text, aug}; audio is decoded lazily in the collator (no 30 GB feature cache)."""

    def __init__(self, rows: Sequence[Dict[str, Any]], audio_dir: str, augment: bool):
        self.items = [{"path": os.path.join(audio_dir, r["audio_file"]), "text": r["ref"], "aug": augment}
                      for r in rows]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


@dataclass
class OnTheFlyCollator:
    """Decode audio, augment train items, extract log-mel features and build masked labels."""
    processor: Any
    augment: Optional[WaveAugment] = None
    seed: int = 0
    _rng: Optional[np.random.Generator] = field(default=None, repr=False)

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        import torch
        if self._rng is None:  # one stream per DataLoader worker
            self._rng = np.random.default_rng(int(torch.initial_seed()) % (2 ** 63))
        audios = []
        for f in features:
            a = load_audio(f["path"])[: int(MAX_AUDIO_SEC * SAMPLE_RATE)]
            if f.get("aug") and self.augment is not None:
                a = self.augment(a, self._rng)
            audios.append(a)
        batch = self.processor.feature_extractor(audios, sampling_rate=SAMPLE_RATE, return_tensors="pt")
        tok = self.processor.tokenizer
        ids = [tok(f["text"]).input_ids for f in features]
        if any(len(x) > MAX_LABEL_TOKENS for x in ids):
            raise ValueError("Label exceeds decoder window; curate before training, never silently truncate")
        width = max(len(x) for x in ids)
        labels = torch.full((len(ids), width), -100, dtype=torch.long)
        for i, x in enumerate(ids):
            labels[i, :len(x)] = torch.tensor(x, dtype=torch.long)
        sot = tok.convert_tokens_to_ids("<|startoftranscript|>")
        if (labels[:, 0] == sot).all():  # the model prepends decoder_start_token_id itself
            labels = labels[:, 1:]
        return {"input_features": batch["input_features"], "labels": labels}


def corpus_wer(refs: Sequence[str], hyps: Sequence[str]) -> float:
    """Corpus WER (%) with the same normalisation as the benchmark."""
    if len(refs) != len(hyps):
        raise ValueError("reference/hypothesis length mismatch")
    err = n = 0
    for r, h in zip(refs, hyps):
        rw, hw = normalize(r).split(), normalize(h).split()
        s, d, i = word_error_counts(rw, hw)
        err, n = err + s + d + i, n + max(len(rw), 1)
    return 100.0 * err / max(n, 1)


# ---------------------------------------------------------------------------
# Model construction (GPU side)
# ---------------------------------------------------------------------------

def from_pretrained(cls, path: str, dtype=None, **kw):
    """transformers renamed torch_dtype -> dtype; accept both versions."""
    try:
        return cls.from_pretrained(path, dtype=dtype, **kw)
    except TypeError:
        return cls.from_pretrained(path, torch_dtype=dtype, **kw)


def set_generation_defaults(model) -> None:
    model.config.use_cache = True
    gc = model.generation_config
    gc.language, gc.task = "indonesian", "transcribe"
    gc.forced_decoder_ids = None
    if hasattr(gc, "use_cache"):
        gc.use_cache = True


def build_lora_model(model_id: str = BASE_MODEL_ID, r: int = 32, alpha: int = 64, dropout: float = 0.05,
                     spec_augment: bool = True, dtype=None):
    """Whisper + LoRA on all attention/MLP projections of encoder and decoder, with SpecAugment on."""
    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import WhisperForConditionalGeneration
    model = from_pretrained(WhisperForConditionalGeneration, model_id, dtype=dtype or torch.float32)
    if spec_augment:
        model.config.apply_spec_augment = True
        model.config.mask_time_prob, model.config.mask_time_length = 0.05, 10
        model.config.mask_feature_prob, model.config.mask_feature_length = 0.05, 10
    set_generation_defaults(model)
    model.config.use_cache = False  # incompatible with gradient checkpointing; re-enabled at export
    model = get_peft_model(model, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout,
                                             target_modules=LORA_TARGETS, bias="none"))
    # conv1 is frozen, so its output has no grad and checkpointed blocks would drop the LoRA gradients
    model.base_model.model.model.encoder.conv1.register_forward_hook(lambda m, i, o: o.requires_grad_(True))
    return model


def make_compute_metrics(tokenizer):
    def compute(pred):
        p = np.where(pred.predictions < 0, tokenizer.pad_token_id, pred.predictions)
        l = np.where(pred.label_ids < 0, tokenizer.pad_token_id, pred.label_ids)
        return {"wer": corpus_wer(tokenizer.batch_decode(l, skip_special_tokens=True),
                                  tokenizer.batch_decode(p, skip_special_tokens=True))}
    return compute


def build_trainer(model, processor, train_ds, dev_ds, out_dir: str, *, epochs: float = 3, lr: float = 1e-4,
                  batch: int = 16, accum: int = 2, eval_steps: int = 250, workers: int = 4, max_steps: int = -1,
                  augment: Optional[WaveAugment] = None, patience: int = 4, seed: int = 0, bf16: Optional[bool] = None):
    """Seq2SeqTrainer selecting the checkpoint with the lowest dev WER (unseen videos)."""
    import math
    import torch
    from transformers import EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments
    cuda = torch.cuda.is_available()
    if bf16 is None:
        bf16 = cuda and torch.cuda.is_bf16_supported()
    total = max_steps if max_steps > 0 else math.ceil(len(train_ds) / (batch * accum)) * epochs
    args = Seq2SeqTrainingArguments(
        output_dir=out_dir, per_device_train_batch_size=batch, per_device_eval_batch_size=batch,
        gradient_accumulation_steps=accum, learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * total)), num_train_epochs=epochs, max_steps=max_steps, weight_decay=0.01,
        bf16=bf16, fp16=cuda and not bf16,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="steps", eval_steps=eval_steps, save_strategy="steps", save_steps=eval_steps,
        save_total_limit=3, load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
        predict_with_generate=True, generation_max_length=225, generation_num_beams=1,
        logging_steps=25, report_to=["none"], remove_unused_columns=False, label_names=["labels"],
        dataloader_num_workers=workers, dataloader_persistent_workers=workers > 0, seed=seed)
    return Seq2SeqTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
        data_collator=OnTheFlyCollator(processor, augment if augment is not None else WaveAugment(), seed),
        compute_metrics=make_compute_metrics(processor.tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)])


def export_merged(peft_model, processor, out_dir: str) -> str:
    """Merge LoRA into the base weights, save fp16 with inference-ready generation defaults."""
    import torch
    model = peft_model.merge_and_unload()
    model.config.apply_spec_augment = False
    set_generation_defaults(model)
    model = model.to(torch.float16)
    model.save_pretrained(out_dir, safe_serialization=True)
    processor.save_pretrained(out_dir)
    return out_dir

In [ ]:
%%writefile test_rme_asr_v2.py
"""CPU tests for the ASR v2 recipe and decoding fallback: python test_rme_asr_v2.py"""
import numpy as np

import benchmark_real as b
import rme_asr_v2 as v2
from local_med_llm import CLINICAL_ONTOLOGY


def _rows(n_videos=40, clips=10, levels=("L2", "L3")):
    return [dict(chunk_id=f"{v}_{i}", audio_file=f"x_{i}_{v}.mp3", dur=10.0, ref="satu dua tiga empat lima",
                 video_id=f"V{v}", level=levels[v % len(levels)], category="c")
            for v in range(n_videos) for i in range(clips)]


def test_split_is_by_video_deterministic_and_stratified():
    rows = _rows()
    s1, s2 = v2.split_videos(rows, 0.1, 0.2, seed=3), v2.split_videos(rows, 0.1, 0.2, seed=3)
    assert s1 == s2 and set(s1) == {r["video_id"] for r in rows}
    for lvl in ("L2", "L3"):  # every level gets its own test and dev videos
        got = {s1[r["video_id"]] for r in rows if r["level"] == lvl}
        assert got == {"train", "dev", "test"}
    test_clips = sum(s1[r["video_id"]] == "test" for r in rows)
    assert 0.15 * len(rows) <= test_clips <= 0.3 * len(rows)
    assert v2.split_videos(rows, 0.1, 0.2, seed=4) != s1


def test_split_counts_clips_not_videos():
    rows = _rows(n_videos=20, clips=5, levels=("L",))
    rows += [dict(r, chunk_id=f"big{i}", video_id="BIG") for i, r in enumerate(_rows(1, 200, ("L",)))]
    s = v2.split_videos(rows, 0.0, 0.2, seed=0)
    test_clips = sum(s[r["video_id"]] == "test" for r in rows)
    # one 200-clip video cannot be followed by more test videos once the clip budget is met
    assert test_clips <= 0.2 * len(rows) + 200


def test_label_issue_reasons():
    ok = dict(ref="pasien mengeluh nyeri dada sejak pagi tadi", dur=4.0)
    assert v2.label_issue(ok) is None
    assert v2.label_issue(dict(ok, dur=45.0)) == "TOO_LONG"
    assert v2.label_issue(dict(ok, dur=0.5)) == "TOO_SHORT"
    assert v2.label_issue(dict(ref="penting adalah memilih tidak kita", dur=13.6)) == "SLOW_LABEL"
    assert v2.label_issue(dict(ref=" ".join(["kata"] + [f"w{i}" for i in range(40)]), dur=5.0)) == "FAST_LABEL"
    assert v2.label_issue(dict(ref="hai hai hai hai hai hai hai hai hai hai", dur=5.0)) == "LOOPING_LABEL"
    assert v2.label_issue(dict(ref="ya", dur=1.5)) == "TOO_FEW_WORDS"
    kept, dropped = v2.curate([ok, dict(ok, dur=45.0)])
    assert len(kept) == 1 and dropped == {"TOO_LONG": 1}


def test_augment_bounded_and_window_safe():
    rng = np.random.default_rng(0)
    aug = v2.WaveAugment(p_speed=1.0, p_noise=1.0, p_gain=1.0)
    x = (0.3 * np.sin(np.linspace(0, 2000, 16000 * 5))).astype(np.float32)
    for _ in range(20):
        y = aug(x, rng)
        assert y.dtype == np.float32 and np.isfinite(y).all() and np.abs(y).max() <= 1.0
        assert 0.85 * len(x) <= len(y) <= 1.15 * len(x)
    long = np.zeros(16000 * 29, dtype=np.float32) + 0.1
    assert all(len(aug(long, rng)) <= 16000 * 30 for _ in range(20))
    assert len(aug(np.zeros(0, dtype=np.float32), rng)) == 0


def test_degenerate_detection():
    assert b.degenerate_reason("terima kasih " * 20, 5.0) == "COMPRESSION"
    assert b.degenerate_reason("pasien datang dengan keluhan nyeri dada kiri sejak dua hari yang lalu", 6.0) is None
    assert b.degenerate_reason("", 5.0) is None                       # silence is fine
    assert b.degenerate_reason("iya iya iya", 2.0) is None              # short natural repetition is fine
    words = " ".join("".join(chr(97 + (i * 7 + j * 3) % 26) for j in range(4 + i % 3)) for i in range(40))
    assert b.degenerate_reason(words, 2.0) == "WORDS_PER_SEC"
    assert b.compression_ratio("abc") < b.compression_ratio("abc " * 50)


def test_decode_profiles_have_internal_flag_only_in_fallback():
    assert b.DECODE_PROFILES["beam_fallback"]["_fallback"] is True
    assert not any(k.startswith("_") for p in ("plain", "guarded") for k in b.DECODE_PROFILES[p])
    assert not any(k.startswith("_") for k in b.FALLBACK_KWARGS)


def test_corpus_wer_matches_benchmark_normalisation():
    assert v2.corpus_wer(["Mata-kanan merah"], ["mata kanan merah"]) == 0.0
    assert abs(v2.corpus_wer(["a b c d"], ["a x c"]) - 50.0) < 1e-9


def test_term_recall_by_group():
    groups = b.term_groups(CLINICAL_ONTOLOGY)
    assert groups["kloramfenikol"].endswith("/medications")
    items = [dict(ref="resep kloramfenikol tetes", hyp="resep kloramfenikol tetes"),
             dict(ref="resep kloramfenikol tetes", hyp="resep klor tetes")]
    g = b.term_recall_by_group(items, groups)[groups["kloramfenikol"]]
    assert g["n"] == 2 and g["hit"] == 1 and g["recall"] == 50.0


if __name__ == "__main__":
    for n in [x for x in dir() if x.startswith("test_")]:
        globals()[n](); print("PASS", n)

In [ ]:
TEST_MODULES = ["test_rme_asr_v2"]
import importlib
failed = []
for name in TEST_MODULES:
    mod = importlib.import_module(name)
    for n in [x for x in dir(mod) if x.startswith("test_")]:
        try:
            getattr(mod, n)(); print("PASS", name, n)
        except Exception as e:
            failed.append(f"{name}.{n}: {e!r}"); print("FAIL", name, n, repr(e))
if failed:
    fail(5, f"{len(failed)} tes gagal: {failed[:3]}", "jangan lanjut; kirim output sel ini ke pemilik proyek")
ok(5, "semua tes logika lulus")

## LANGKAH 6/10 — Split per-video dan kurasi label
- **Tujuan:** membagi video train/dev/test dan membuang label rusak; split disimpan ke Drive
- **Perkiraan waktu:** ±1 menit
- **Berhasil jika:** `✅ LANGKAH 6 OK` dan tercetak jumlah video train/dev/test
- **Jika gagal:** ikuti TINDAKAN yang dicetak

In [ ]:
STEP = 6
import collections, hashlib
import benchmark_real as bm, rme_asr_v2 as v2
load = lambda f: bm.normalize_rows(json.load(open(f)))
v1_train, v1_test = load("grand_train_manifest.json"), load("grand_test_manifest.json")
ALL = v1_train + v1_test
manifest_md5 = hashlib.md5(open("grand_train_manifest.json", "rb").read() + open("grand_test_manifest.json", "rb").read()).hexdigest()
if os.path.exists(SPLIT_PATH):
    saved = json.load(open(SPLIT_PATH))
    if saved["manifest_md5"] != manifest_md5:
        fail(STEP, "manifest berubah sejak split dibuat", "jangan lanjut; laporkan ke pemilik proyek (split lama tidak boleh dicampur data baru)")
    SPLIT = saved["video_split"]; print("Split dimuat dari", SPLIT_PATH)
else:
    SPLIT = v2.split_videos(ALL, DEV_FRAC, TEST_FRAC, SPLIT_SEED)
by = collections.defaultdict(list)
for r in ALL: by[SPLIT[r["video_id"]]].append(r)
for s in ("train", "dev", "test"):
    rs = by[s]; print(f"{s:5s}: {len({r['video_id'] for r in rs}):3d} video, {len(rs):5d} klip, {sum(r['dur'] for r in rs)/3600:5.1f} jam,", dict(collections.Counter(r['level'][:7] for r in rs)))
if {r["video_id"] for r in by["train"]} & {r["video_id"] for r in by["test"] + by["dev"]}:
    fail(STEP, "video yang sama ada di train dan test/dev", "jangan lanjut; laporkan ke pemilik proyek")

have = set(os.listdir("data/all_audio"))
train_rows, CURATION = v2.curate([r for r in by["train"] if r["audio_file"] in have])
dev_rows = bm.stratified_sample([r for r in by["dev"] if r["audio_file"] in have and bm.eligible(r)], DEV_EVAL_CLIPS, seed=0)
print("Train dipakai:", len(train_rows), "| dibuang:", CURATION, "| dev:", len(dev_rows))
if len(train_rows) < 15000 or len(dev_rows) < 100:
    fail(6, "data latih/dev terlalu sedikit", "ulangi LANGKAH 4 (audio belum lengkap)")
json.dump({"manifest_md5": manifest_md5, "seed": SPLIT_SEED, "dev_frac": DEV_FRAC, "test_frac": TEST_FRAC,
           "video_split": SPLIT}, open(SPLIT_PATH, "w"))
ok(6, f"split disimpan ke {SPLIT_PATH}")

## LANGKAH 7/10 — Training
- **Tujuan:** melatih LoRA v2; checkpoint ke Drive dan dilanjutkan otomatis jika sesi putus
- **Perkiraan waktu:** A100 ±1,5–2,5 jam (3 epoch) · T4 ±3–4 jam (1 epoch)
- **Berhasil jika:** `✅ LANGKAH 7 OK — dev WER terbaik ...`
- **Jika gagal:** jika sesi putus: sambungkan lagi dan Run all (training lanjut dari checkpoint). Jika `CUDA out of memory`: ubah `TRAIN_PROFILE` di LANGKAH 2 menjadi `"hemat"` lalu Run all

In [ ]:
import glob, time
from transformers import WhisperProcessor
proc = WhisperProcessor.from_pretrained(v2.BASE_MODEL_ID, language="indonesian", task="transcribe")
model = v2.build_lora_model(v2.BASE_MODEL_ID, r=LORA_R, alpha=LORA_ALPHA)
model.print_trainable_parameters()
train_ds = v2.ClipDataset(train_rows, "data/all_audio", augment=True)
dev_ds = v2.ClipDataset(dev_rows, "data/all_audio", augment=False)
trainer = v2.build_trainer(model, proc, train_ds, dev_ds, CKPT_DIR, epochs=EPOCHS, lr=LR, batch=BATCH, accum=ACCUM,
                           eval_steps=EVAL_STEPS, workers=WORKERS, max_steps=MAX_STEPS, patience=PATIENCE)
resume = bool(glob.glob(f"{CKPT_DIR}/checkpoint-*"))
print("Lanjut dari checkpoint di" if resume else "Mulai baru di", CKPT_DIR)
t0 = time.time(); trainer.train(resume_from_checkpoint=resume or None); TRAIN_S = time.time() - t0
hist = [x for x in trainer.state.log_history if "eval_wer" in x]
for x in hist: print(f"step {x['step']:5d} | dev WER {x['eval_wer']:.2f}% | eval loss {x['eval_loss']:.4f}")
if not hist or trainer.state.best_metric is None:
    fail(7, "tidak ada evaluasi dev yang tercatat", "kirim output sel ini ke pemilik proyek")
ok(7, f"dev WER terbaik {trainer.state.best_metric:.2f}% ({trainer.state.best_model_checkpoint})")

## LANGKAH 8/10 — Ekspor model merge ke Drive
- **Tujuan:** menggabungkan LoRA ke model dan menyimpan .tar.gz ke Drive
- **Perkiraan waktu:** ±5 menit
- **Berhasil jika:** `✅ LANGKAH 8 OK`
- **Jika gagal:** pastikan Drive punya ruang kosong ≥ 3 GB, lalu jalankan ulang sel ini

In [ ]:
import gc, shutil
v2.export_merged(trainer.model, proc, MERGED_DIR)
!cd /content && tar czf whisper_large_v3_turbo_rme_v2_rc_merged.tar.gz whisper_large_v3_turbo_rme_v2_rc_merged
shutil.copy("/content/whisper_large_v3_turbo_rme_v2_rc_merged.tar.gz", DRIVE_DIR)
if os.path.getsize(f"{DRIVE_DIR}/whisper_large_v3_turbo_rme_v2_rc_merged.tar.gz") < 1e9:
    fail(8, "file model di Drive terlalu kecil (salinan tidak lengkap)", "jalankan ulang sel ini")
del trainer, model; gc.collect(); torch.cuda.empty_cache()
ok(8, f"{DRIVE_DIR}/whisper_large_v3_turbo_rme_v2_rc_merged.tar.gz")

## LANGKAH 9/10 — Cek kewajaran di video dev: base vs v2
- **Tujuan:** memastikan v2 lebih baik dari model dasar sebelum benchmark resmi
- **Perkiraan waktu:** ±10–15 menit
- **Berhasil jika:** `✅ LANGKAH 9 OK — v2 lebih baik dari base di dev`
- **Jika gagal:** jika v2 tidak lebih baik: jangan jalankan benchmark; kirim output ke pemilik proyek

In [ ]:
DEV_CHECK = {}
for name, path in (("base", None), ("rme_v2", MERGED_DIR)):
    pipe = bm.build_asr_pipe(path)
    for pf in ("guarded", "beam_fallback"):
        h, sp = bm.transcribe_rows(pipe, dev_rows, "data/all_audio", 16, WORKERS, profile=pf)
        items = [dict(ref=r["ref"], hyp=h[r["chunk_id"]], dur=r["dur"]) for r in dev_rows if h.get(r["chunk_id"]) is not None]
        m = bm.score_items(items, [], B=300)
        DEV_CHECK[f"{name}|{pf}"] = {"wer": m["wer"], "wer_ci95": m["wer_ci95"], "x_realtime": sp["x_realtime_gpu"],
                                     "fallback_flagged": sp["fallback_flagged"], "fallback_fixed": sp["fallback_fixed"]}
        print(f"{name:7s} {pf:13s} dev WER {m['wer']:.2f}% (CI {m['wer_ci95'][0]:.1f}-{m['wer_ci95'][1]:.1f}) | {sp['x_realtime_gpu']:.0f}x RT | fallback {sp['fallback_flagged']}->{sp['fallback_fixed']}")
    del pipe; gc.collect(); torch.cuda.empty_cache()
DEV_OK = DEV_CHECK["rme_v2|guarded"]["wer"] < DEV_CHECK["base|guarded"]["wer"]
if not DEV_OK:
    print("❌ v2 TIDAK lebih baik dari base di dev. Jangan jalankan benchmark; kirim output ini ke pemilik proyek.")
else:
    ok(9, "v2 lebih baik dari base di dev")

## LANGKAH 10/10 — Laporan dan status akhir
- **Tujuan:** menyimpan laporan training ke Drive dan mencetak status akhir
- **Perkiraan waktu:** <1 menit
- **Berhasil jika:** baris terakhir `STATUS_TRAINING: OK`
- **Jika gagal:** jika `STATUS_TRAINING: GAGAL`, kirim seluruh output notebook ke pemilik proyek

In [ ]:
report = {"status": "OK" if DEV_OK else "GAGAL", "gpu": GPU, "profile": profile, "train_seconds": TRAIN_S,
          "train_clips": len(train_rows), "curation_dropped": CURATION, "dev_clips": len(dev_rows),
          "config": dict(EPOCHS=EPOCHS, LR=LR, LORA_R=LORA_R, LORA_ALPHA=LORA_ALPHA, BATCH=BATCH, ACCUM=ACCUM,
                         LORA_TARGETS=v2.LORA_TARGETS, SPLIT_SEED=SPLIT_SEED),
          "eval_history": hist, "dev_check": DEV_CHECK}
json.dump(report, open(f"{DRIVE_DIR}/rme_v2_rc_training_report.json", "w"), indent=1)
print(json.dumps({k: v for k, v in report.items() if k != "eval_history"}, indent=1))
print(f"STATUS_TRAINING: {report['status']} | profil {profile} | dev WER v2 {DEV_CHECK['rme_v2|guarded']['wer']:.2f}% vs base {DEV_CHECK['base|guarded']['wer']:.2f}%")
if DEV_OK: print("Berikutnya: buka benchmark_v2_colab_t4.ipynb dengan runtime T4, lalu Run all.")